# 💬 딸깍콘 (DdalkakCon)
### AI 기반 움직이는 이모티콘 제작 프로그램

**2026 인공지능 인재양성 프로그램 — 심화역량과정**

| 역할 | 성명 | 담당 |
|---|---|---|
| 함수 코드 제작, 서버 및 배포 | 강은석 | `emoticon_tools.py` · `geo_motion.py` · `geo_effects.py` |
| 프론트엔드 및 코드 통합 | 이은석 | `app.py` |

---

### 처리 흐름

```
사용자 이미지 + 프롬프트
   │
   ▼  LangChain / gpt-4o-mini   프롬프트 해석, 프레임 기획 (tool calling)
   │
   ▼  OpenAI 이미지 API  (호출 1회)
   ①  스프라이트 시트 1장 생성     ← 6~8프레임을 한 장에. 같은 붓질이라 색이 안 흔들린다
   ②  칸 분할 + 좌우 정렬
   │
   ▼  Pillow (AI 호출 없음, 비용 0)
   ③  투명 배경 안정화            노이즈 제거 + 내부 구멍 메우기
   ④  세트 공통 배율 + 규격       360×360, 종마다 크기가 같아지게
   ⑤  공통 배율로 규격 배치        종마다 크기가 같아지게
   ⑥  효과·글자 합성             하트/반짝임/눈물/음표 + 한글
   ⑦  저장                       GIF + 번호 붙인 PNG 프레임
   │
   ▼
카카오 WebP Animator 에 넣으면 제출 완료
```

### 차별점
- **이음새 없는 루프** — 프레임 간 변화량을 측정해 끊김을 수치로 잡아냄
- **투명 배경 안정화** — 생성 시점부터 투명 + 점 노이즈·내부 구멍 후처리
- **규격 자동 충족** — 360×360 / 24프레임 이하 / 용량 자동 조정
- **스프라이트 시트** — 한 번의 생성으로 6~8프레임. 프레임 간 색 표준편차 R 0.5
  (따로 그리면 R 4.8, 진짜 카카오 이모티콘은 R 1.5)
- **팔다리가 실제로 움직인다** — 손 흔들기, 꾸벅 졸기 가능

## 1. 설치 (라이브러리 + 한글 폰트)

In [ ]:
%pip install -q langchain langchain-openai openai streamlit pillow
# 한글 폰트 — 이모티콘에 글자를 넣으려면 필요하다.
# 나눔고딕은 OFL 라이선스라 상업적 사용이 허용된다 (OGQ 반려 사유 중 하나가 폰트 라이선스).
!apt-get install -y -qq fonts-nanum > /dev/null 2>&1
!ls /usr/share/fonts/truetype/nanum/ | head -3

## 2. API 키 등록

Colab 왼쪽 🔑(보안 비밀)에 `openai` 라는 이름으로 키를 저장하고, **노트북 액세스 토글을 켠 뒤** 실행한다.

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('openai')
print("API 키 등록 완료:", os.environ["OPENAI_API_KEY"][:7] + "...")

## 3. 핵심 파이프라인 — `emoticon_tools.py`
**담당: 강은석** — 이미지 API 호출, 투명 배경 처리, 규격 맞추기, 저장, LangChain tool

In [ ]:
%%writefile emoticon_tools.py
# -*- coding: utf-8 -*-
"""
딸깍콘(DdalkakCon) - 움직이는 이모티콘 생성 핵심 함수 모듈
담당: 함수 코드 (백엔드)

[파이프라인]
  (1) 기준 원화(base sheet) 1장 확정        : OpenAI 이미지 API
  (2) 프레임 N장 '병렬' 생성 ((1)을 레퍼런스): OpenAI 이미지 API + ThreadPoolExecutor
  (3) 투명 배경 안정화 (opening/closing)     : Pillow
  (4) 프레임 공통 바운딩박스 정렬            : Pillow  <- 떨림 제거, 의도된 모션은 보존
  (5) 규격 맞추기 (360x360 캔버스)           : Pillow
  (6) 무한 루핑 시퀀스 (핑퐁)                : 프레임 재배열
  (7) 저장: GIF(공용 팔레트) + WebP(알파)    : Pillow

[설계 의도 - 보고서/발표에 쓸 포인트]
  - 프레임을 따로따로 생성하면 캐릭터가 프레임마다 달라진다.
    -> (1)에서 '기준 원화' 1장을 먼저 확정하고, 모든 프레임이 그 원화만 참조하게 해서 일관성을 확보한다.
  - 프레임마다 캐릭터를 각각 중앙정렬하면 '점프' 같은 의도된 움직임이 사라진다.
    -> (4)에서 전체 프레임의 '합집합' 바운딩박스 하나를 구해 모든 프레임에 똑같이 적용한다.
  - GIF는 프레임별로 팔레트를 따로 만들면 재생 중 색이 깜빡인다.
    -> (7)에서 전체 프레임으로 공용 팔레트 1개를 만들어 모든 프레임이 공유한다.
"""

from __future__ import annotations

import base64
import io
import os
import time
from concurrent.futures import ThreadPoolExecutor
from typing import List, Literal, Optional, Tuple

from PIL import Image, ImageChops, ImageDraw, ImageFilter
from langchain_core.tools import tool

# ---------------------------------------------------------------- 설정 상수

IMAGE_MODEL = "gpt-image-1.5"   # gpt-image-1은 2026-10-23 은퇴 예정 자료가 있어 1.5를 기본값으로 둔다
GEN_SIZE = "1024x1024"          # 생성 해상도 (360보다 크게 뽑아 축소 -> 외곽선이 깔끔해진다)

CANVAS = 360                    # 카카오 이모티콘 스튜디오 '움직이는 이모티콘' 제안 규격 (px)
MAX_FRAMES = 24                 # 제안 규격: 24프레임 이하
# 용량 한도 (공식 가이드 확인값)
KAKAO_MAX_BYTES = 650 * 1024    # 카카오 '움직이는 이모티콘' WebP 1개당
SAFE_MAX_BYTES = 256 * 1024     # 전 플랫폼 교집합 (디스코드 이모지가 가장 빡셈)
MAX_BYTES = SAFE_MAX_BYTES      # 기본값은 어디든 통과하는 쪽으로 둔다

OUT_DIR = os.environ.get("DDALKAK_OUT_DIR", "/content")
GIF_PATH = os.path.join(OUT_DIR, "generated_emoticon.gif")
SHEET_PATH = os.path.join(OUT_DIR, "generated_sheet.png")   # 스프라이트 시트 원본
WEBP_PATH = os.path.join(OUT_DIR, "generated_emoticon.webp")
BASE_PATH = os.path.join(OUT_DIR, "generated_base.png")
FRAME_PATH_FMT = os.path.join(OUT_DIR, "generated_frame_{:02d}.png")

# 이미지 생성 AI에 공통으로 붙이는 스타일 고정 문구
STYLE_SUFFIX = (
    "2D cute emoticon sticker style, bold clean outlines, flat vivid colors, "
    "fully transparent background, single character centered, "
    "no background elements, no ground shadow, "
    "no checkerboard pattern, no border, no frame. "
    # 효과와 글자는 Pillow가 정확한 위치에 직접 그린다.
    # AI가 자기 멋대로 넣으면 우리 효과와 겹쳐서 두 종류가 섞인다.
    "Draw ONLY the character itself: absolutely no text, no letters, no speech bubbles, "
    "no sparkles, no stars, no hearts, no tears, no sweat drops, no motion lines, "
    "no emotion symbols and no decorative effects of any kind around the character"
)

# ------------------------------------------------- 업로드 이미지 공유 슬롯
# LangChain tool은 LLM이 만든 인자만 받으므로, 사용자가 올린 원본 이미지는
# 앱(app.py)이 아래 슬롯에 넣어두고 tool이 꺼내 쓰는 구조로 전달한다.

_reference_image: Optional[bytes] = None
_reference_mime: str = "image/png"

# 기준 원화 캐시. 대화 중 이모티콘을 여러 개 만들어도 같은 캐릭터가 나오게 한다.
# (없으면 매번 새 캐릭터가 생겨서 세트로 쓸 수 없다)
_base_cache: Optional[Image.Image] = None

# 마지막 생성 결과 메타데이터 (UI 표시용)
last_result: dict = {}


def set_reference_image(data: Optional[bytes], mime: str = "image/png") -> None:
    """app.py가 사용자 업로드 이미지를 등록한다. 바뀌면 기준 원화 캐시를 버린다."""
    global _reference_image, _reference_mime, _base_cache
    if data != _reference_image:
        _base_cache = None            # 다른 캐릭터이므로 원화를 다시 만들어야 한다
    _reference_image = data
    _reference_mime = mime or "image/png"


def has_reference_image() -> bool:
    return _reference_image is not None


# ------------------------------------------------------------ OpenAI 호출

def _client():
    """OpenAI 클라이언트. OPENAI_API_KEY 환경변수를 사용한다."""
    from openai import OpenAI
    return OpenAI()


def _ext_for(mime: str) -> str:
    return {
        "image/png": "png",
        "image/jpeg": "jpg",
        "image/jpg": "jpg",
        "image/webp": "webp",
    }.get(mime, "png")


def _as_upload(data: bytes, mime: str = "image/png", name: str = "ref") -> Tuple:
    """openai SDK가 받는 (파일명, 바이트, MIME) 형태로 감싼다."""
    return (f"{name}.{_ext_for(mime)}", io.BytesIO(data), mime)


def _supports_input_fidelity(model: str) -> bool:
    # gpt-image-1 / gpt-image-1.5 만 input_fidelity 지원 (gpt-image-2 계열은 미지원)
    return model in ("gpt-image-1", "gpt-image-1.5")


def _call_image_api(prompt: str, refs: Optional[List[Tuple]] = None,
                    quality: str = "low", size: Optional[str] = None) -> bytes:
    """
    이미지 1장 생성. refs가 있으면 edit(레퍼런스 기반), 없으면 generate.
    size 를 주면 그 크기로 뽑는다 (스프라이트 시트는 1536x1024 등을 쓴다).
    """
    client = _client()
    common = dict(
        model=IMAGE_MODEL,
        prompt=prompt,
        size=size or GEN_SIZE,
        quality=quality,
        background="transparent",   # 생성 시점부터 투명하게 뽑는다 (사후 제거보다 외곽선이 안전)
        output_format="png",        # background="transparent"는 png/webp에서만 유효
        n=1,
    )
    if refs:
        if _supports_input_fidelity(IMAGE_MODEL):
            common["input_fidelity"] = "high"   # 레퍼런스 캐릭터를 최대한 유지
        result = client.images.edit(image=refs, **common)
    else:
        result = client.images.generate(**common)
    return base64.b64decode(result.data[0].b64_json)


# -------------------------------------------------- (1) 기준 원화 만들기

def build_base_sheet(character_description: str, quality: str = "low") -> Image.Image:
    """
    모든 프레임이 참조할 '기준 원화' 1장을 만든다.
    업로드 이미지가 있으면 그 캐릭터를 이모티콘 스타일로 정규화하고,
    없으면 설명만으로 새로 생성한다.
    """
    if _reference_image is not None:
        prompt = (
            "Redraw the character in the reference image as a clean emoticon sticker. "
            "Keep its identity, colors, proportions and distinctive features exactly. "
            f"{character_description}. Neutral front-facing idle pose. {STYLE_SUFFIX}"
        )
        refs = [_as_upload(_reference_image, _reference_mime, "character")]
        raw = _call_image_api(prompt, refs=refs, quality=quality)
    else:
        prompt = f"{character_description}. Neutral front-facing idle pose. {STYLE_SUFFIX}"
        raw = _call_image_api(prompt, refs=None, quality=quality)

    img = Image.open(io.BytesIO(raw)).convert("RGBA")
    _save_png(img, BASE_PATH)
    return img


def get_base_sheet(character_description: str, quality: str = "low") -> Image.Image:
    """
    기준 원화를 가져온다. 처음 한 번만 만들고 이후에는 재사용한다.

    대화 중 이모티콘을 여러 개 만들 때 매번 새로 생성하면 캐릭터가 조금씩
    달라져서 세트로 쓸 수 없다. 업로드 이미지가 바뀌면 캐시는 자동으로 버려진다.
    """
    global _base_cache
    if _base_cache is None:
        _base_cache = build_base_sheet(character_description, quality=quality)
    return _base_cache


# ----------------------------------------- (2) 프레임 N장 병렬 생성

def build_pose_prompt(pose: str) -> str:
    """
    기준 원화를 참조해 '포즈와 표정만' 바꾸는 프롬프트.
    프레임 생성과 종별 대표 포즈 생성이 같은 문구를 쓰도록 한 곳에 모았다.

    글자는 넣지 않는다 — Pillow가 정확한 위치에 직접 쓴다 (STYLE_SUFFIX 참고).
    """
    return (
        "Use the reference image as the exact character design. "
        "Keep the same character, same art style, same colors, same line weight, "
        "same camera distance and the same canvas framing. "
        f"Change only the pose and expression to: {pose}. {STYLE_SUFFIX}"
    )


def generate_pose(base_bytes: bytes, pose: str, quality: str = "low",
                  retries: int = 2, name: str = "base") -> Image.Image:
    """
    기준 원화를 레퍼런스로 포즈 1장 생성. 실패하면 잠깐 쉬었다 다시 시도한다.

    이미지 API는 일시적 오류(429/5xx)와 경계선 검열 판정이 섞여 나온다.
    한 번 실패했다고 포기하면 세트 전체가 구멍 난다.
    """
    last = None
    for attempt in range(retries + 1):
        try:
            raw = _call_image_api(build_pose_prompt(pose),
                                  refs=[_as_upload(base_bytes, "image/png", name)],
                                  quality=quality)
            return Image.open(io.BytesIO(raw)).convert("RGBA")
        except Exception as exc:       # noqa: BLE001 - 사유를 그대로 올려보낸다
            last = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise last


# ====================================================== 스프라이트 시트
# 프레임을 한 장씩 따로 생성하면 그릴 때마다 색과 선이 미세하게 달라져서
# 재생할 때 캐릭터가 부글거린다(boiling). 실측으로 색 표준편차가 R 4.8 / B 6.6 이었고,
# 진짜 카카오 이모티콘은 R 1.5 / B 2.0 이었다.
#
# 그래서 '한 번의 생성'으로 여러 프레임을 한 장에 그리게 한다.
# 같은 붓질 안에서 나오므로 색·선·비율이 원리적으로 같아진다.
# 실측 결과 색 표준편차가 R 0.5 / B 0.4 까지 떨어졌다 (진짜 이모티콘보다 안정적).

# 칸 수 -> (가로칸, 세로칸, 생성 크기)
# gpt-image-1.5 는 1024x1024 / 1536x1024 / 1024x1536 만 지원한다.
# gpt-image-2 는 임의 크기를 지원하지만 '투명 배경'을 지원하지 않아 쓸 수 없다.
SHEET_LAYOUTS = {
    4: (2, 2, "1024x1024"),     # 칸 512x512  정사각
    6: (3, 2, "1536x1024"),     # 칸 512x512  정사각
    8: (4, 2, "1536x1024"),     # 칸 384x512  가로가 좁다
}
# 칸이 정사각이라 가로로 퍼지는 자세도 담기는 격자
SQUARE_CELL_LAYOUTS = (4, 6)


def pick_layout(n_steps: int, spread: str = "narrow") -> Tuple[int, int, str, int]:
    """
    원하는 프레임 수에 가장 가까운 격자를 고른다. (가로칸, 세로칸, 크기, 실제칸수)

    spread="wide" 면 칸이 정사각인 격자만 쓴다.

    왜 필요한가:
      4x2 격자는 칸이 384x512 로 가로가 좁다. 서 있거나 팔을 드는 자세는 세로로
      길쭉해서 문제없지만, 넘어지거나 눕는 자세는 가로로 퍼져서 칸을 넘친다.
      실제로 '뒤로 넘어지는' 8프레임 결과물에서 분할선이 캐릭터를 가로질러
      조각났다(면적이 1번 칸 대비 52%까지 떨어졌다). 같은 동작을 3x2(512x512)로
      뽑았을 때는 멀쩡했다.
    """
    allowed = SQUARE_CELL_LAYOUTS if spread == "wide" else tuple(SHEET_LAYOUTS)
    n = min(allowed, key=lambda k: (abs(k - n_steps), -k))
    cols, rows, size = SHEET_LAYOUTS[n]
    return cols, rows, size, n


def build_sheet_prompt(character_description: str, steps: List[str],
                       cols: int, rows: int) -> str:
    """
    스프라이트 시트 생성 프롬프트.

    핵심 지시 세 가지
      1. 모든 칸이 '같은 캐릭터, 같은 색, 같은 선 굵기'
      2. 모든 칸에서 캐릭터가 '같은 크기, 칸 안 같은 위치' (카메라 고정)
      3. 칸 구분선·번호·글자 없음
    """
    n = cols * rows
    lines = "\n".join(f"  cell {i + 1}: {s}" for i, s in enumerate(steps[:n]))
    return (
        f"Create ONE image that is a {cols}x{rows} sprite sheet of animation frames "
        f"({cols} columns, {rows} rows, {n} cells of equal size).\n"
        f"The character is: {character_description}.\n"
        "Use the reference image as the EXACT character design. Keep the identical "
        "character, identical colors, identical line weight and identical art style "
        "in every single cell.\n"
        "Every cell must show the character at the SAME size and the SAME position "
        "within its own cell, as if the camera never moves. Only the pose changes.\n"
        # 칸을 넘치면 분할선이 캐릭터를 가로질러 조각난다
        "Keep the whole character well inside its own cell, leaving an empty margin "
        "of at least 15% on every side of the cell. The character must never touch "
        "or cross a cell boundary, even in the widest pose. If a pose is wide, draw "
        "the character smaller so it still fits with that margin.\n"
        "The cells form a strict uniform grid with no visible separators.\n"
        f"Frame order is left to right, then top to bottom:\n{lines}\n"
        f"{STYLE_SUFFIX}"
    )


def check_cells(cells: List[Image.Image], min_area_ratio: float = 0.6,
                max_blobs: int = 3) -> List[str]:
    """
    분할된 칸이 멀쩡한지 검사한다. 문제가 있으면 사유 목록을 돌려준다.

    왜 필요한가:
      모델이 격자를 안 지키면 분할선이 캐릭터를 가로질러 반 토막이 나고,
      옆 칸 조각이 섞여 들어온다. 그런데 지금까지는 그걸 그대로 GIF 로 만들어
      사용자에게 줬다. 실제로 면적이 1번 칸 대비 52%까지 떨어진 결과물이
      아무 경고 없이 나갔다.

    두 가지를 본다
      1. 면적 : 중앙값 대비 min_area_ratio 미만이면 캐릭터가 잘린 것
      2. 조각 : 떨어진 덩어리가 많으면 옆 칸이 비쳐 들어온 것

    '칸 가장자리에 닿았는가'는 쓰지 않는다. 멀쩡한 시트도 캐릭터가 칸을 꽉 채우면
    가장자리에 닿기 때문에 오탐이 심했다(정상 시트 2장에서 10건이 잘못 잡혔다).
    문제는 '닿는 것'이 아니라 '잘리는 것'이다.
    """
    import numpy as np

    if len(cells) < 2:
        return ["칸이 2개 미만"]

    masks = [np.asarray(c.getchannel("A")) > 128 for c in cells]
    areas = [int(m.sum()) for m in masks]
    med = float(np.median(areas)) or 1.0

    problems = []
    for i, a in enumerate(areas, 1):
        if a < med * min_area_ratio:
            problems.append(f"{i}번 칸 면적 {a/med*100:.0f}% (캐릭터가 잘림)")

    try:
        from scipy import ndimage
        for i, (m, a) in enumerate(zip(masks, areas), 1):
            lab, cnt = ndimage.label(m)
            if cnt <= 1:
                continue
            sizes = ndimage.sum(m, lab, range(1, cnt + 1))
            big = int((sizes > a * 0.05).sum())     # 아주 작은 점은 세지 않는다
            if big > max_blobs:
                problems.append(f"{i}번 칸에 조각 {big}개 (옆 칸이 비쳐 들어옴)")
    except ImportError:
        pass                                        # scipy 없으면 조각 검사만 생략

    return problems


MAX_TOTAL_FRAMES = 24          # 카카오 '움직이는 이모티콘' 상한


def plan_sheets(n_frames: int, spread: str = "narrow") -> List[int]:
    """
    N프레임을 시트 몇 장에 나눠 담을지 정한다. 각 시트의 칸 수 목록을 돌려준다.

    격자가 전부 2행이라 한 장에서 나오는 칸 수는 4·6·8 뿐이고 모두 짝수다.
    그래서 15·21 같은 홀수는 한 장으로 못 만들고, 마지막 시트에서 남는 칸을 버린다.

    예) 15 narrow -> [8, 8]      (16칸 중 15칸 사용, API 2회)
        21 narrow -> [8, 8, 8]  (24칸 중 21칸 사용, API 3회)

    모든 시트가 '같은 격자'가 되게 고른다. 섞으면 칸 모양이 달라져서 모델이
    캐릭터를 다른 크기로 그린다. 실제로 12프레임을 [8,4](4x2 + 2x2)로 뽑았더니
    뒤쪽 4장이 1.85배 커져서 재생할 때 캐릭터가 갑자기 부풀었다.
    """
    import math

    n_frames = max(2, min(MAX_TOTAL_FRAMES, n_frames))
    opts = SQUARE_CELL_LAYOUTS if spread == "wide" else tuple(SHEET_LAYOUTS)

    best = None
    for k in opts:
        sheets = math.ceil(n_frames / k)
        waste = sheets * k - n_frames
        score = (sheets, waste)          # 호출 수를 먼저, 그다음 버리는 칸을 줄인다
        if best is None or score < best[0]:
            best = (score, [k] * sheets)
    return best[1]


def generate_sprite_sheet(character_description: str, steps: List[str],
                          quality: str = "high", spread: str = "narrow",
                          retries: int = 2, extra_refs: Optional[List[bytes]] = None
                          ) -> Tuple[Image.Image, int, int, int, List[str]]:
    """
    스프라이트 시트를 생성하고 분할이 멀쩡한지 확인한다.
    (시트, 가로칸, 세로칸, 칸수, 경고목록)

    분할이 깨지면 '더 큰 칸'을 쓰는 격자로 자동 재시도한다. 칸이 커지면
    프레임 수는 줄지만, 토막 난 이모티콘을 내보내는 것보다 낫다.
    업로드 이미지가 있으면 그것을 레퍼런스로 쓰고, 없으면 설명만으로 그린다.
    """
    refs = []
    if _reference_image is not None:
        refs.append(_as_upload(_reference_image, _reference_mime, "character"))
    # 앞선 시트의 칸을 같이 보여주면 장 사이에서 캐릭터가 덜 달라진다
    for i, extra in enumerate(extra_refs or []):
        refs.append(_as_upload(extra, "image/png", f"prev{i}"))
    refs = refs or None

    want = len(steps)
    plan = [spread, "wide"] if spread != "wide" else ["wide"]
    if 4 not in SQUARE_CELL_LAYOUTS:
        plan.append("wide")
    notes: List[str] = []
    last_err = None
    best = None                                     # 다 실패하면 그나마 나은 것

    for round_no, sp in enumerate(plan):
        cols, rows, size, n = pick_layout(want, spread=sp)
        prompt = build_sheet_prompt(character_description, steps, cols, rows)
        for attempt in range(retries + 1):
            try:
                raw = _call_image_api(prompt, refs=refs, quality=quality, size=size)
                sheet = Image.open(io.BytesIO(raw)).convert("RGBA")
            except Exception as exc:                # noqa: BLE001
                last_err = exc
                if attempt < retries:
                    time.sleep(1.5 * (attempt + 1))
                continue

            cells = split_sheet(sheet, cols, rows)
            bad = check_cells(cells)
            if not bad:
                _save_png(sheet, SHEET_PATH)
                return sheet, cols, rows, n, notes
            if best is None:
                best = (sheet, cols, rows, n, bad)
            last_err = None
            break                                   # 생성은 됐으니 격자를 바꿔 재시도

        if round_no == 0 and best is not None:
            notes.append(f"{cols}x{rows} 격자에서 분할이 깨져 "
                         f"더 큰 칸으로 다시 생성했습니다 ({best[4][0]})")

    if best is not None:
        sheet, cols, rows, n, bad = best
        _save_png(sheet, SHEET_PATH)
        notes.append("격자가 완전하지는 않습니다: " + "; ".join(bad[:3]))
        return sheet, cols, rows, n, notes
    raise last_err or RuntimeError("스프라이트 시트 생성 실패")


def split_sheet(sheet: Image.Image, cols: int, rows: int,
                min_fill: float = 0.01) -> List[Image.Image]:
    """시트를 칸으로 자른다. 거의 비어 있는 칸은 버린다."""
    import numpy as np

    w, h = sheet.size
    cw, ch = w // cols, h // rows
    out = []
    for r in range(rows):
        for c in range(cols):
            cell = sheet.crop((c * cw, r * ch, (c + 1) * cw, (r + 1) * ch))
            if (np.asarray(cell.getchannel("A")) > 200).mean() < min_fill:
                continue
            out.append(cell)
    return out


def align_frames(cells: List[Image.Image], band: float = 0.40,
                 limit_ratio: float = 0.35) -> List[Image.Image]:
    """
    칸마다 캐릭터가 좌우로 어긋난 것을 맞춘다.

    모델이 칸 안에서 캐릭터를 늘 같은 자리에 두지는 않는다. 실측으로 132px
    (칸 폭의 34%)까지 어긋났고, 그대로 재생하면 캐릭터가 좌우로 덜컹거린다.

    방법: 아래쪽 band 부분(몸통·다리)의 '열별 픽셀 수' 1차원 프로파일을
          1번 프레임과 상호상관시켜 가장 잘 맞는 가로 이동량을 찾는다.
          팔은 움직이므로 위쪽은 보지 않는다.

    왜 이 방법인가 (8가지를 같은 기준으로 비교한 결과)
      보정 없음 0.409 / bbox 중심 0.567 / 하단 겹침최대 0.739 / 이 방법 0.738
      품질은 겹침최대와 같은데 9배 빠르다 (24ms vs 211ms).
    세로는 실측 1px 밖에 안 어긋나 보정하지 않는다.
    """
    import numpy as np

    if len(cells) < 2:
        return list(cells)

    h = cells[0].size[1]
    y0 = int(h * (1 - band))
    limit = int(cells[0].size[0] * limit_ratio)

    def profile(img):
        a = np.asarray(img.getchannel("A"))[y0:, :] > 128
        p = a.sum(axis=0).astype(float)
        return p - p.mean()

    ref = profile(cells[0])
    out = [cells[0]]
    for cell in cells[1:]:
        cur = profile(cell)
        best, shift = -1e18, 0
        for s in range(-limit, limit + 1):
            v = float((ref * np.roll(cur, s)).sum())
            if v > best:
                best, shift = v, s
        if shift == 0:
            out.append(cell)
            continue
        arr = np.roll(np.asarray(cell), shift, axis=1)
        if shift > 0:
            arr[:, :shift] = 0
        else:
            arr[:, shift:] = 0
        out.append(Image.fromarray(arr, "RGBA"))
    return out


def stabilize_alpha(img: Image.Image, threshold: int = 24) -> Image.Image:
    """
    알파 채널을 정리한다.
      - threshold 미만의 희미한 알파를 완전 투명으로 (반투명 찌꺼기 제거)
      - opening(MinFilter3 -> MaxFilter3) 으로 떠다니는 점 노이즈 제거
      - closing(MaxFilter5 -> MinFilter5) 으로 4px 이하 바늘구멍 메우기
    큰 구멍은 fill_interior_holes()가 따로 처리한다.
    """
    img = img.convert("RGBA")
    alpha = img.getchannel("A")
    alpha = alpha.point(lambda v: 0 if v < threshold else v)
    alpha = alpha.filter(ImageFilter.MinFilter(3)).filter(ImageFilter.MaxFilter(3))
    alpha = alpha.filter(ImageFilter.MaxFilter(5)).filter(ImageFilter.MinFilter(5))
    img.putalpha(alpha)
    return img


def robust_bbox(img: Image.Image, frac: float = 0.01
                ) -> Optional[Tuple[int, int, int, int]]:
    """
    점 노이즈에 흔들리지 않는 바운딩박스.

    Pillow의 getbbox()는 '알파가 0이 아닌 픽셀'을 전부 포함한다. 그래서
    캐릭터와 멀리 떨어진 작은 점 하나가 박스를 수백 픽셀 부풀리고,
    그만큼 캐릭터가 쓸데없이 작게 축소된다. 생성 AI 결과에는 그런
    찌꺼기가 흔히 남는다.

    여기서는 행/열별 '알파 질량'을 구해, 최댓값의 frac 배를 넘는 행/열만
    경계로 인정한다. 점 노이즈는 질량이 작아 자동으로 빠지고 캐릭터
    윤곽은 남는다. frac=0.01 에서 실측 오차 1px 수준이었고, 0.005 이하로
    내리면 노이즈가 다시 포함됐다.
    """
    import numpy as np

    a = np.asarray(img.convert("RGBA").getchannel("A"), dtype=np.float64)
    if a.sum() <= 0:
        return None

    def span(v):
        peak = v.max()
        if peak <= 0:
            return None
        idx = np.nonzero(v > peak * frac)[0]
        if idx.size == 0:
            return None
        return int(idx[0]), int(idx[-1]) + 1

    sx = span(a.sum(axis=0))
    sy = span(a.sum(axis=1))
    if sx is None or sy is None:
        return img.convert("RGBA").getchannel("A").getbbox()
    return (sx[0], sy[0], sx[1], sy[1])


def fill_interior_holes(img: Image.Image, cut: int = 128) -> Image.Image:
    """
    '테두리와 연결되지 않은' 투명 영역만 불투명으로 메운다.

    이미지 생성 API가 캐릭터 안쪽의 밝은 영역(눈 흰자, 치아, 밝은 옷 등)을
    배경으로 오인해 뚫어버리는 현상이 보고되어 있다. 형태학적 closing은
    작은 구멍만 닫을 수 있어서, 캔버스 바깥 테두리에서 flood fill을 돌려
    '바깥 배경'을 식별하고 그 밖에 남은 투명 영역을 내부 구멍으로 판정한다.
    캐릭터 겉 실루엣은 건드리지 않는다.
    """
    img = img.convert("RGBA")
    w, h = img.size
    binary = img.getchannel("A").point(lambda v: 255 if v >= cut else 0)

    # 1px 투명 테두리를 둘러 바깥 배경이 항상 (0,0)과 이어지게 만든다
    padded = Image.new("L", (w + 2, h + 2), 0)
    padded.paste(binary, (1, 1))
    ImageDraw.floodfill(padded, (0, 0), 128, thresh=0)   # 바깥 배경만 128로 표시
    outside = padded.crop((1, 1, w + 1, h + 1))

    # 128(바깥)만 투명으로 두고, 나머지(캐릭터 + 내부 구멍)는 불투명으로
    filled = outside.point(lambda v: 0 if v == 128 else 255)

    alpha = img.getchannel("A")
    img.putalpha(ImageChops.lighter(alpha, filled))
    return img


def _median_char_height(cells: List[Image.Image]) -> float:
    """칸들에서 캐릭터 세로 높이의 중앙값."""
    hs = []
    for c in cells:
        b = robust_bbox(c)
        if b:
            hs.append(b[3] - b[1])
    if not hs:
        return 0.0
    hs.sort()
    return float(hs[len(hs) // 2])


def _match_height(cells: List[Image.Image], ref_h: float) -> List[Image.Image]:
    """
    이 시트의 캐릭터 높이를 기준 높이에 맞춘다.
    장마다 크기가 다르면 재생할 때 캐릭터가 갑자기 커졌다 작아진다.
    """
    cur = _median_char_height(cells)
    if cur <= 0 or abs(cur - ref_h) / ref_h < 0.04:
        return cells
    k = ref_h / cur
    out = []
    for c in cells:
        w, h = c.size
        nw, nh = max(1, int(w * k)), max(1, int(h * k))
        sheet = Image.new("RGBA", (w, h), (0, 0, 0, 0))
        sheet.paste(c.resize((nw, nh), Image.LANCZOS),
                    ((w - nw) // 2, (h - nh) // 2))
        out.append(sheet)
    return out


def generate_frames_multi(character_description: str, steps: List[str],
                          quality: str = "high", spread: str = "narrow"
                          ) -> Tuple[List[Image.Image], List[str], List[str]]:
    """
    필요한 만큼 스프라이트 시트를 여러 장 만들어 프레임을 모은다.
    (프레임 리스트, 시트별 '가로x세로' 표기, 경고 목록)

    한 장에 8칸까지만 들어가므로 12·15·24프레임 같은 요청은 여러 장이 필요하다.
    두 번째 장부터는 '첫 장의 1번 칸'을 레퍼런스에 같이 넣어 캐릭터가 바뀌지 않게 한다.
    (장마다 따로 그리면 장 사이에서 색과 비율이 달라진다)
    """
    plan = plan_sheets(len(steps), spread=spread)
    frames: List[Image.Image] = []
    grids: List[str] = []
    warns: List[str] = []

    anchor_cell: Optional[bytes] = None      # 첫 장의 1번 칸 — 이후 장의 기준
    ref_h = 0.0                              # 첫 장의 캐릭터 높이 — 크기 기준

    pos = 0
    for idx, want in enumerate(plan):
        chunk = steps[pos:pos + want]
        pos += want
        if not chunk:
            break
        # 칸이 남으면 마지막 자세로 채운다 (빈 칸은 모델이 멋대로 그린다)
        while len(chunk) < want:
            chunk.append(chunk[-1])

        # 2장째부터는 원본 레퍼런스에 '첫 장의 1번 칸'을 덧붙인다.
        # 원본을 교체해버리면 원래 색과 질감 기준을 잃는다.
        extra = [anchor_cell] if anchor_cell is not None else None
        sheet, cols, rows, n, w = generate_sprite_sheet(
            character_description, chunk, quality=quality, spread=spread,
            extra_refs=extra)
        warns.extend(w)
        grids.append(f"{cols}x{rows}")

        cells = split_sheet(sheet, cols, rows)
        cells = align_frames(cells)
        if idx == 0 and cells:
            anchor_cell = _to_png_bytes(cells[0])
            ref_h = _median_char_height(cells)
        elif cells and ref_h:
            # 격자를 통일해도 장마다 캐릭터 크기가 조금씩 다르다. 1장 기준으로 맞춘다
            cells = _match_height(cells, ref_h)
        frames.extend(cells)

    want_total = len(steps)
    if len(frames) > want_total:
        frames = frames[:want_total]              # 남는 칸은 버린다
    elif len(frames) < want_total:
        warns.append(f"요청 {want_total}프레임 중 {len(frames)}프레임만 얻었습니다")
    return frames, grids, warns


def save_with_budget(frames: List[Image.Image], path: str, durations,
                     ) -> Tuple[int, List[str]]:
    """
    용량 한도를 적응적으로 적용해 저장한다.

    프레임이 많으면 256KB(전 플랫폼 교집합)를 못 맞춘다. 그때 색을 32색까지
    깎으면 화질이 무너지는데도 한도를 못 넘는 경우가 있었다(24프레임 실측 267KB).
    그래서 먼저 256KB 를 노리고, 안 되면 색을 되돌리고 카카오 한도(650KB)로
    올린 뒤 사용자에게 알린다.
    """
    notes: List[str] = []
    size, note = shrink_until_under_limit(frames, path, durations,
                                          limit=SAFE_MAX_BYTES)
    if size <= SAFE_MAX_BYTES:
        if note:
            notes.append(note)
        return size, notes

    size = save_gif(frames, path=path, duration_ms=durations)   # 색 복원
    if size <= KAKAO_MAX_BYTES:
        notes.append(
            f"{size/1024:.0f}KB — 카카오({KAKAO_MAX_BYTES//1024}KB)는 통과하지만 "
            f"전 플랫폼 공통 한도({SAFE_MAX_BYTES//1024}KB)는 넘습니다. "
            f"디스코드·라인에 쓰려면 프레임을 줄이세요")
        return size, notes

    size, note = shrink_until_under_limit(frames, path, durations,
                                          limit=KAKAO_MAX_BYTES)
    notes.append(f"용량이 커서 색을 줄였습니다 ({size/1024:.0f}KB). "
                 f"프레임 수를 줄이는 쪽이 화질에 낫습니다")
    return size, notes


# ------------------- (4)+(5) 공통 바운딩박스 정렬 + 규격 캔버스

def _canvas_wh(canvas) -> Tuple[int, int]:
    """canvas 는 정수(정사각형) 또는 (가로, 세로) 둘 다 받는다."""
    if isinstance(canvas, (tuple, list)):
        return int(canvas[0]), int(canvas[1])
    return int(canvas), int(canvas)


def union_bbox(frames: List[Image.Image]) -> Optional[Tuple[int, int, int, int]]:
    """여러 프레임을 모두 덮는 bbox. 효과 기준점을 '고정'하는 데 쓴다."""
    boxes = [robust_bbox(f) for f in frames]
    boxes = [b for b in boxes if b]
    if not boxes:
        return None
    return (min(b[0] for b in boxes), min(b[1] for b in boxes),
            max(b[2] for b in boxes), max(b[3] for b in boxes))


def _layout(side: int, canvas, margin: float,
            reserve_top: float, reserve_bottom: float):
    """
    배치 계산을 한 곳에 모은 것. 프레임 묶음과 종 세트가 같은 수식을 쓴다.

    reserve_top / reserve_bottom 은 글자나 효과를 올릴 자리를 위/아래에
    비워두는 비율(0~1)이다. 안 비우면 캐릭터가 캔버스를 꽉 채워
    글자가 얼굴을 덮는다.

    돌려주는 값: (캔버스 가로세로, 캐릭터를 그릴 한 변, 좌상단 오프셋)
    """
    cw, ch = _canvas_wh(canvas)
    avail_w = cw * (1 - 2 * margin)
    avail_h = ch * (1 - 2 * margin - reserve_top - reserve_bottom)
    if avail_w <= 1 or avail_h <= 1:
        raise ValueError("여백/예약 공간이 너무 커서 캐릭터를 놓을 자리가 없습니다")
    target = max(1, int(min(avail_w, avail_h)))
    ox = (cw - target) // 2
    oy = int(ch * (margin + reserve_top)) + max(0, (int(avail_h) - target) // 2)
    return (cw, ch), target, (ox, oy)


def _place(img: Image.Image, center, side: int, target: int,
           offset, size) -> Image.Image:
    """center 를 중심으로 한 변 side 인 정사각형을 떠서 캔버스에 붙인다."""
    cx, cy = center
    x0, y0 = cx - side // 2, cy - side // 2
    crop = img.crop((x0, y0, x0 + side, y0 + side))   # 경계를 넘어도 투명으로 채워진다
    sheet = Image.new("RGBA", size, (0, 0, 0, 0))
    sheet.paste(crop.resize((target, target), Image.LANCZOS), offset)
    return sheet


def fit_frames_to_canvas(frames: List[Image.Image], canvas=CANVAS,
                         margin_ratio: float = 0.03,
                         reserve_top: float = 0.0,
                         reserve_bottom: float = 0.0) -> List[Image.Image]:
    """
    '한 애니메이션의 프레임들'을 캔버스에 맞춘다.

    모든 프레임에 똑같은 크롭/스케일을 적용한다. 프레임별로 따로 중앙정렬하면
    점프 같은 의도된 모션이 사라지기 때문이다.
    canvas 는 정수 또는 (가로, 세로) — OGQ 740x640, 라인 320x270 대응.
    """
    if not frames:
        return []
    size = _canvas_wh(canvas)
    box = union_bbox(frames)
    if box is None:                      # 전부 투명이면 축소만 한다
        return [f.resize(size, Image.LANCZOS) for f in frames]

    left, top, right, bottom = box
    side = max(right - left, bottom - top)
    center = ((left + right) // 2, (top + bottom) // 2)
    size, target, offset = _layout(side, canvas, margin_ratio,
                                   reserve_top, reserve_bottom)
    return [_place(f, center, side, target, offset, size) for f in frames]


def fit_art_set(arts: dict, canvas=(CANVAS, CANVAS), margin: float = 0.04,
                reserve_top: float = 0.0, reserve_bottom: float = 0.0) -> dict:
    """
    '서로 다른 종'의 그림을 같은 배율로 배치한다.
    {라벨: 그림} -> {라벨: (배치된 그림, 배치된 캐릭터 bbox)}

    fit_frames_to_canvas() 를 종마다 따로 쓰면 안 되는 이유:
      그건 프레임 간 상대 위치를 보존하려고 만든 함수다. 서로 다른 종에 각각
      적용하면 팔 든 포즈와 움츠린 포즈가 제각각 확대돼서 24종의 캐릭터
      크기가 들쭉날쭉해진다. 그래서 여기서는 가장 큰 종을 기준으로 배율을
      하나 정하고 위치만 종별로 맞춘다.

    돌려주는 bbox 는 '실제로 배치된' 캐릭터 영역이라
    geo_effects.anchors_from_bbox() 에 그대로 넘기면 된다.
    """
    size = _canvas_wh(canvas)
    boxes = {k: robust_bbox(v) for k, v in arts.items()}
    boxes = {k: b for k, b in boxes.items() if b}
    if not boxes:
        return {k: (v.resize(size, Image.LANCZOS), (0, 0) + size)
                for k, v in arts.items()}

    side = max(max(b[2] - b[0], b[3] - b[1]) for b in boxes.values())
    size, target, offset = _layout(side, canvas, margin, reserve_top, reserve_bottom)

    out = {}
    for k, img in arts.items():
        b = boxes.get(k)
        if not b:
            out[k] = (img.resize(size, Image.LANCZOS), (0, 0) + size)
            continue
        center = ((b[0] + b[2]) // 2, (b[1] + b[3]) // 2)
        sheet = _place(img, center, side, target, offset, size)
        out[k] = (sheet, robust_bbox(sheet) or offset + (offset[0] + target,
                                                         offset[1] + target))
    return out


def fit_art_for_motion(art: Image.Image, motion: str, canvas=(CANVAS, CANVAS),
                       margin: float = 0.045, reserve_top: float = 0.0,
                       max_up_ratio: float = 0.22):
    """
    '모션이 움직일 공간'과 '글자·효과 자리'를 먼저 빼고 캐릭터를 배치한다.

    왜 필요한가:
      점프는 캐릭터를 95px 위로 올리고, 낙하는 110px 올린다. 캐릭터를 먼저
      크게 배치해놓고 모션을 걸면 머리가 캔버스 밖으로 잘리거나 글자와 겹친다.

    처리 순서
      1. 모션이 필요한 여백(위/아래/좌우)과 늘어남 배율을 미리 구한다
      2. 위로 가는 양이 캔버스의 max_up_ratio 를 넘으면 '움직임'을 줄인다
         (캐릭터를 더 줄이는 것보다 낫다 - 캐릭터가 너무 작아지면 안 예쁘다)
      3. 남은 공간에 캐릭터를 넣고 바닥 쪽에 붙인다

    돌려주는 값: (배치된 그림, 캐릭터 bbox, 모션에 적용할 배율)
    """
    import math

    import geo_motion as gm

    w, h = _canvas_wh(canvas)
    params = gm.MOTIONS[motion]()
    mx, my = w * margin, h * margin

    # 위로 가는 양을 캔버스의 max_up_ratio 로 제한 (넘치면 움직임을 줄인다)
    up_raw = max(0.0, -min(p.get("dy", 0.0) for p in params))
    mscale = min(1.0, h * max_up_ratio / up_raw) if up_raw > 1e-6 else 1.0

    down = max(0.0, max(p.get("dy", 0.0) for p in params)) * mscale
    bottom = h - my - down                      # 캐릭터 발바닥이 놓일 높이
    top_limit = my + h * reserve_top            # 그 위는 글자·효과 자리
    room = bottom - top_limit

    # 프레임마다 '그 순간' 필요한 크기를 따로 계산해 가장 빡빡한 것에 맞춘다.
    # 방향별 최댓값을 따로 구해서 곱하면(늘어남 최대 x 높이 최대) 실제로는
    # 동시에 일어나지 않는 상황까지 피하느라 캐릭터가 쓸데없이 작아진다.
    side = float(min(room, w - 2 * mx))
    for p in params:
        th = math.radians(p.get("rot", 0.0))
        spread = abs(math.cos(th)) + abs(math.sin(th))
        sy = max(1e-3, p.get("sy", 1.0) * spread)
        sx = max(1e-3, p.get("sx", 1.0) * spread)
        up_f = max(0.0, -p.get("dy", 0.0) * mscale)
        dx_f = abs(p.get("dx", 0.0)) * mscale
        side = min(side, (room - up_f) / sy)                  # 위로 안 잘리게
        side = min(side, 2 * (w / 2 - mx - dx_f) / sx)        # 좌우로 안 잘리게

    target = int(max(32, side))

    box = robust_bbox(art)
    if box is None:
        sheet = art.resize((w, h), Image.LANCZOS)
        return sheet, (0, 0, w, h), mscale

    src_side = max(box[2] - box[0], box[3] - box[1])
    center = ((box[0] + box[2]) // 2, (box[1] + box[3]) // 2)
    ox = (w - target) // 2
    oy = int(bottom) - target
    sheet = _place(art, center, src_side, target, (ox, oy), (w, h))
    return sheet, (robust_bbox(sheet) or (ox, oy, ox + target, oy + target)), mscale


# ----------------------------------------- (6) 무한 루핑 시퀀스

def build_loop_sequence(frames: List[Image.Image], mode: str = "pingpong") -> List[Image.Image]:
    """
    mode="pingpong" : 1-2-3-4-3-2-1 로 재배열
    mode="cycle"    : 1-2-3-4-1 (동작 자체가 이미 순환하는 경우)

    왜 '대표 프레임(1번)'으로 끝나야 하는가 — 세 플랫폼 규칙이 여기서 동시에 풀린다.
      카카오 : 마지막 프레임이 이모티콘샵/키보드 썸네일로 쓰인다
      라인   : 첫 프레임이 정적 이미지로 노출된다
      OGQ    : 첫 프레임과 마지막 프레임이 같아야 한다
    따라서 대표 포즈를 1번에 두고 1번으로 돌아오면 셋 다 만족한다.
    (예전엔 1-2-3-4-3-2 로 끝나서 어중간한 중간 동작이 썸네일이 됐다)
    """
    if len(frames) < 2:
        return list(frames)

    if mode == "cycle":
        seq = list(frames) + [frames[0]]
    elif len(frames) < 3:
        seq = list(frames) + [frames[0]]
    else:
        seq = list(frames) + list(reversed(frames[1:-1])) + [frames[0]]

    if len(seq) > MAX_FRAMES:       # 제안 규격: 24프레임 이하
        seq = seq[:MAX_FRAMES - 1] + [frames[0]]   # 잘라도 대표로 끝맺는다
    return seq


# ------------------------------------------------------- (7) 저장

def _to_png_bytes(img: Image.Image) -> bytes:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return buf.getvalue()


def _save_png(img: Image.Image, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    img.save(path, format="PNG")


def _hard_mask(img: Image.Image, cut: int = 128) -> Image.Image:
    """GIF는 1비트 투명만 지원하므로 알파를 0/255로 이진화한다."""
    return img.getchannel("A").point(lambda v: 255 if v >= cut else 0)


def _build_shared_palette(frames: List[Image.Image], colors: int = 255) -> Image.Image:
    """
    전체 프레임을 한 장으로 이어붙여 공용 팔레트를 만든다.
    프레임마다 팔레트를 따로 만들면 재생 중 색이 깜빡이는데, 그걸 막는다.
    colors=255 로 두어 인덱스 255번을 '투명'용으로 비워둔다.
    """
    w, h = frames[0].size
    montage = Image.new("RGB", (w, h * len(frames)), (255, 255, 255))
    for i, f in enumerate(frames):
        flat = Image.new("RGB", (w, h), (255, 255, 255))
        flat.paste(f.convert("RGB"), (0, 0), _hard_mask(f))
        montage.paste(flat, (0, h * i))
    return montage.convert("P", palette=Image.ADAPTIVE, colors=colors)


def _reduce_palette(master: Image.Image, colors: int, sample: int = 6144) -> Image.Image:
    """
    이미 만든 공용 팔레트를 '더 적은 색'으로 줄인다.

    용량이 넘칠 때마다 몽타주(수백만 픽셀)를 다시 양자화하면 매번 ~95ms가 든다.
    여기서는 팔레트 색을 '실제 사용 빈도만큼 반복한' 수천 픽셀짜리 띠를 만들어
    그것만 양자화한다. 빈도를 반영하므로 화질 손해는 작고 속도는 수십 배 빠르다.
    """
    hist = master.histogram()[:256]
    total = sum(hist) or 1
    data = []
    for idx, cnt in enumerate(hist):
        if cnt <= 0:
            continue
        data.extend([idx] * max(1, round(cnt / total * sample)))
    if not data:
        return master

    strip = Image.new("P", (len(data), 1))
    strip.putpalette(master.getpalette())
    strip.putdata(data)
    return strip.convert("RGB").convert("P", palette=Image.ADAPTIVE, colors=colors)


def save_gif(frames: List[Image.Image], path: str = GIF_PATH,
             duration_ms=150, colors: int = 255,
             master: Optional[Image.Image] = None) -> int:
    """
    투명 배경 + 무한 루프 GIF 저장. 저장된 바이트 크기를 반환한다.

    duration_ms : 정수(전 프레임 동일) 또는 프레임별 리스트
    master      : 미리 만들어둔 공용 팔레트 (없으면 여기서 만든다)
    """
    if master is None:
        master = _build_shared_palette(frames, colors=colors)
    transparent_idx = 255

    pal_frames = []
    for f in frames:
        mask = _hard_mask(f)
        flat = Image.new("RGB", f.size, (255, 255, 255))
        flat.paste(f.convert("RGB"), (0, 0), mask)
        p = flat.quantize(palette=master, dither=Image.Dither.NONE)
        p.paste(transparent_idx, mask=ImageChops.invert(mask))  # 배경을 투명 인덱스로
        pal_frames.append(p)

    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    pal_frames[0].save(
        path,
        format="GIF",
        save_all=True,
        append_images=pal_frames[1:],
        duration=duration_ms,
        loop=0,                     # 0 = 무한 반복
        transparency=transparent_idx,
        disposal=2,                 # 다음 프레임 전에 지우기 (투명 영역 잔상 방지)
        optimize=False,
    )
    return os.path.getsize(path)


def save_webp(frames: List[Image.Image], path: str = WEBP_PATH,
              duration_ms: int = 150) -> int:
    """
    알파가 온전히 살아있는 애니메이션 WebP 저장.
    카카오는 '승인 후' 단계에서 WebP를 요구하므로 미리 같이 내보낸다.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    frames[0].save(
        path,
        format="WEBP",
        save_all=True,
        append_images=frames[1:],
        duration=duration_ms,
        loop=0,
        lossless=True,
    )
    return os.path.getsize(path)


def shrink_until_under_limit(frames: List[Image.Image], path: str,
                             duration_ms=150, limit: int = MAX_BYTES) -> Tuple[int, str]:
    """
    용량 한도를 넘으면 색 수를 단계적으로 줄여 다시 저장한다.
    프레임을 버리면 움직임이 끊기므로 색을 먼저 깎는다.

    공용 팔레트는 '한 번만' 만들고, 재시도에서는 _reduce_palette() 로 줄인다.
    """
    master = _build_shared_palette(frames, colors=255)
    size = 0
    for colors in (255, 128, 64, 32):
        pal = master if colors == 255 else _reduce_palette(master, colors)
        size = save_gif(frames, path=path, duration_ms=duration_ms, master=pal)
        if size <= limit:
            note = (f"용량 한도({limit//1024}KB)를 맞추려고 색을 {colors}색으로 줄였습니다"
                    if colors != 255 else "")
            return size, note
    return size, (f"32색으로 줄여도 {size/1024:.0f}KB "
                  f"(한도 {limit//1024}KB 초과) — 프레임 수를 줄여야 합니다")


# =========================================================== LangChain Tool

@tool(parse_docstring=True)
def create_animated_emoticon(
    character_description: str,
    frame_steps: List[str],
    pose_spread: Literal["narrow", "wide"] = "narrow",
    effect: Literal["none", "heart", "sparkle", "tear",
                    "sweat", "anger", "note"] = "none",
    overlay_text: str = "",
    speed_ms: int = 110,
    hold_last_ms: int = 300,
    quality: Literal["low", "medium", "high"] = "high",
) -> str:
    """움직이는 이모티콘 파일(GIF)을 실제로 만든다.

    사용자가 이모티콘 제작을 명확히 요청했을 때만 호출한다. 기획안만 보여달라고 하면
    호출하지 말고 대화로만 답한다. character_description 과 frame_steps 는 영문으로 쓴다.

    한 번의 생성으로 모든 프레임을 한 장(스프라이트 시트)에 그리기 때문에, 프레임마다
    색과 선이 흔들리지 않는다. 손 흔들기, 꾸벅 졸기처럼 팔다리가 따로 움직이는 동작도
    된다. 같은 대화 안에서 여러 번 호출하면 캐릭터가 유지된다.

    Args:
        character_description: 캐릭터의 외형, 색상, 분위기를 묘사한 영문 문장. 사용자가 이미지를 올렸다면 그 캐릭터를 유지하면서 보강할 설명을 쓴다.
        frame_steps: 프레임별 자세를 묘사한 영문 문장 리스트. 최소 4개에서 최대 24개까지 가능하다. 6개면 가장 빠르고 싸며(생성 1회, 약 45초), 12개는 2회, 24개는 3~4회가 든다. 사용자가 프레임 수를 말하지 않으면 6개 또는 8개로 한다. 동작이 복잡하거나 사용자가 부드럽게 해달라고 하면 12개 이상으로 늘린다. 15개를 넘으면 파일이 커져서 카카오 외 플랫폼 기준을 넘길 수 있다. 각 문장은 '한 장의 정지 그림'을 묘사하며 움직임을 설명하지 않는다. 동작이 자연스럽게 이어지도록 순서대로 쓰고, 반복 재생되므로 첫 번째와 마지막은 같은 기본 자세로 둔다. 마지막 자세가 이모티콘샵 썸네일이 되므로 캐릭터를 가장 잘 보여주는 자세로 한다. 효과나 글자는 쓰지 않는다. 예시는 front paw down at its side with a calm smile / front paw lifted to chest height / front paw raised high beside the head with an open smile 처럼 쓴다.
        pose_spread: 동작이 가로로 얼마나 퍼지는지 고른다. narrow 는 서 있거나 팔을 드는 것처럼 세로로 길쭉한 동작이다. wide 는 넘어지기, 눕기, 구르기, 대자로 뻗기처럼 가로로 넓게 퍼지는 동작이다. wide 를 고르면 칸을 크게 잡아 그림이 잘리지 않는다. 대신 프레임 수가 최대 6개로 줄어든다.
        effect: 덧붙일 효과를 고른다. none 은 효과 없음이다. heart 는 하트가 떠올라 설렘에 쓴다. sparkle 은 반짝임으로 최고나 뿌듯함에 쓴다. tear 는 눈물로 슬픔에 쓴다. sweat 는 땀방울로 당황에 쓴다. anger 는 김이 뿜어져 화남에 쓴다. note 는 음표가 떠올라 신남에 쓴다.
        overlay_text: 이모티콘에 넣을 짧은 한글 글자. 필요 없으면 빈 문자열로 둔다. ㅋㅋㅋ 고마워 미안 같이 짧을수록 좋다. 글자는 폰트로 정확히 찍히므로 그림에 맡기지 않는다.
        speed_ms: 프레임 하나의 재생 시간을 밀리초로 준다. 80에서 150 사이를 권장하고 작을수록 빠르다.
        hold_last_ms: 마지막 프레임에서 더 머무는 시간을 밀리초로 준다. 동작이 끝나고 잠깐 쉬는 느낌을 준다. 0이면 쉬지 않는다.
        quality: 그림 품질이다. high 를 권장한다. 빠르게 확인만 할 때는 low 를 쓴다.
    """
    import geo_effects as gfx

    canvas = (CANVAS, CANVAS)
    if len(frame_steps) < 2:
        return "오류: frame_steps 가 너무 적습니다. 6개 또는 8개를 주세요."

    try:
        # (1) 스프라이트 시트 1장 생성 — AI 호출은 여기 한 번뿐
        cells, grids, warns = generate_frames_multi(
            character_description, frame_steps, quality=quality,
            spread=pose_spread)
        if len(cells) < 2:
            return ("오류: 격자가 제대로 생성되지 않았습니다. 다시 시도해 보세요."
                    + ("\n사유: " + "; ".join(warns) if warns else ""))

        # (4) 투명 배경 정리
        cleaned = [stabilize_alpha(c) for c in cells]

        # (5) 공통 배율로 규격 캔버스에 배치. 글자가 있으면 위를 비운다
        reserve = 0.20 if overlay_text else (0.08 if effect != "none" else 0.0)
        frames = fit_frames_to_canvas(cleaned, canvas=canvas, margin_ratio=0.05,
                                      reserve_top=reserve)
        frames = [fill_interior_holes(f) for f in frames]

        # (6) 효과와 글자 — Pillow가 캐릭터 위치에 맞춰 직접 그린다
        box = union_bbox(frames)
        if box and (effect != "none" or overlay_text):
            anchors = gfx.anchors_from_bbox(box, canvas)
            if effect != "none":
                frames = gfx.composite(frames, gfx.EFFECTS[effect](
                    n=len(frames), anchors=anchors, canvas=canvas))
            if overlay_text:
                frames = gfx.composite(frames, gfx.fx_text(
                    overlay_text, n=len(frames), anchors=anchors, canvas=canvas))

        # (7) 재생 시간 — 마지막 프레임에서 잠깐 쉰다
        durations = [speed_ms] * len(frames)
        durations[-1] = min(2000, speed_ms + max(0, hold_last_ms))

        # (8) 저장 — GIF + 번호 붙인 PNG 프레임(카카오 WebP Animator 용)
        gif_size, size_notes = save_with_budget(frames, GIF_PATH, durations)
        warns.extend(size_notes)
        webp_size = save_webp(frames, WEBP_PATH, durations)

        frames_dir = os.path.join(OUT_DIR, "frames")
        os.makedirs(frames_dir, exist_ok=True)
        for i, f in enumerate(frames, 1):
            f.save(os.path.join(frames_dir, f"{i:02d}.png"))

    except Exception as exc:   # LLM이 사유를 보고 다시 시도할 수 있게 문자열로 돌려준다
        return f"이모티콘 생성 실패: {type(exc).__name__}: {exc}"

    last_result.update(
        gif_path=GIF_PATH, webp_path=WEBP_PATH, sheet_path=SHEET_PATH,
        frames_dir=frames_dir,
        frame_paths=[SHEET_PATH],          # AI가 그린 원본 = 스프라이트 시트 한 장
        unique_frames=len(frames), total_frames=len(frames),
        loop_mode=f"시트 {len(grids)}장 ({'+'.join(grids)})", effect=effect,
        duration_ms=speed_ms, gif_size=gif_size, webp_size=webp_size,
    )

    total_s = sum(durations) / 1000
    dropped = (f" (요청 {len(frame_steps)}칸 중 {len(frames)}칸 사용)"
               if len(frames) != len(frame_steps) else "")
    return (
        f"이모티콘 생성 완료!\n"
        f"- 스프라이트 시트 {len(grids)}장({' + '.join(grids)})을 생성해 "
        f"{len(frames)}프레임으로 분할{dropped}\n"
        f"- 재생 {total_s:.1f}초 (프레임당 {speed_ms}ms, 마지막 {durations[-1]}ms)\n"
        + (f"- 효과: {effect}\n" if effect != "none" else "")
        + (f"- 글자: {overlay_text}\n" if overlay_text else "")
        + f"- 규격: {CANVAS}x{CANVAS}px, 투명 배경, GIF {gif_size/1024:.0f}KB\n"
        + "".join(f"- 주의: {w}\n" for w in warns)
        + "화면 왼쪽 사이드바에서 미리보기와 다운로드가 가능하다고 사용자에게 안내하세요."
    )


TOOLS = [create_animated_emoticon]
TOOL_DICT = {t.name: t for t in TOOLS}

## 4. 모션 모듈 — `geo_motion.py`
**담당: 강은석** — 그림 1장에 위치·기울기·눌림을 수식으로 입혀 움직임을 만든다. 모션 12종 + 이징 7종.

In [ ]:
%%writefile geo_motion.py
# -*- coding: utf-8 -*-
"""
기하 변형 모션 모듈

AI가 그린 프레임에 '위치 / 기울기 / 눌림·늘어남'을 수식으로 입혀 움직임을 만든다.
신경망 보간을 쓰지 않으므로 선이 깨지지 않고 알파가 안전하다.

핵심 3가지
  (1) 앵커(anchor) : 무엇을 기준으로 변형하는가.
                     통통 튀기는 '발바닥'이 기준이어야 바닥에 붙어 보인다.
  (2) 이징(easing) : 어떤 속도 곡선으로 변하는가. 선형은 로봇처럼 보인다.
  (3) 홀드(hold)   : '멈춤'을 프레임 복제가 아니라 '그 프레임의 재생시간'으로 만든다.
                     카카오 WebP Animator가 프레임별 0.05~2.0초를 지원하므로,
                     정점에서 멈칫하는 연출에 프레임을 낭비하지 않는다.
"""

from __future__ import annotations

import math
from typing import Callable, Dict, List, Tuple

from PIL import Image

# ------------------------------------------------------------------ 이징 곡선


def linear(t: float) -> float:
    return t


def ease_in(t: float) -> float:
    """느리게 시작해 가속. 떨어지는 것."""
    return t * t


def ease_out(t: float) -> float:
    """빠르게 시작해 감속. 던져진 것이 멈출 때."""
    return 1 - (1 - t) ** 2


def ease_in_out(t: float) -> float:
    """느리게-빠르게-느리게. 가장 무난한 기본값."""
    return 2 * t * t if t < 0.5 else 1 - (-2 * t + 2) ** 2 / 2


def back_out(t: float, s: float = 1.70158) -> float:
    """목표를 살짝 지나쳤다 돌아옴(오버슈트). '뿅' 등장."""
    return 1 + (s + 1) * (t - 1) ** 3 + s * (t - 1) ** 2


def bounce_out(t: float) -> float:
    """바닥에 닿고 통통 튕김."""
    n, d = 7.5625, 2.75
    if t < 1 / d:
        return n * t * t
    if t < 2 / d:
        t -= 1.5 / d
        return n * t * t + 0.75
    if t < 2.5 / d:
        t -= 2.25 / d
        return n * t * t + 0.9375
    t -= 2.625 / d
    return n * t * t + 0.984375


def elastic_out(t: float) -> float:
    """고무줄처럼 진동하며 멈춤."""
    if t in (0.0, 1.0):
        return t
    c = (2 * math.pi) / 3
    return 2 ** (-10 * t) * math.sin((t * 10 - 0.75) * c) + 1


EASINGS: Dict[str, Callable[[float], float]] = {
    "linear": linear, "ease_in": ease_in, "ease_out": ease_out,
    "ease_in_out": ease_in_out, "back_out": back_out,
    "bounce_out": bounce_out, "elastic_out": elastic_out,
}


# ------------------------------------------------------- 변형 1장 적용 (핵심)

def transform_frame(
    img: Image.Image,
    dx: float = 0.0, dy: float = 0.0,
    sx: float = 1.0, sy: float = 1.0,
    rot: float = 0.0,
    anchor: Tuple[float, float] = (0.5, 1.0),
    **_ignored,
) -> Image.Image:
    """
    이동/확대축소/회전을 한 번에 적용한다. 캔버스 크기는 유지된다.
    Pillow의 AFFINE은 '출력 → 입력' 역방향 매핑을 요구하므로 역행렬을 직접 넣는다.

    anchor : 변형 기준점 (0~1 비율). (0.5, 1.0) = 하단 중앙 = 발바닥
    """
    img = img.convert("RGBA")
    w, h = img.size
    ax, ay = anchor[0] * w, anchor[1] * h

    th = math.radians(rot)
    cos, sin = math.cos(th), math.sin(th)
    sx, sy = max(sx, 1e-6), max(sy, 1e-6)
    tx, ty = ax + dx, ay + dy

    a = cos / sx
    b = sin / sx
    c = (-cos * tx - sin * ty) / sx + ax
    d = -sin / sy
    e = cos / sy
    f = (sin * tx - cos * ty) / sy + ay

    return img.transform((w, h), Image.AFFINE, (a, b, c, d, e, f),
                         resample=Image.BICUBIC, fillcolor=(0, 0, 0, 0))


def _squash(q: float) -> Tuple[float, float]:
    """부피 보존 눌림. q>1 세로로 늘어남, q<1 납작. 가로를 반대로 보정한다."""
    return (1.0 / math.sqrt(q), q)


def _kf(keys: List[Tuple[float, float, float]], amp: float) -> List[dict]:
    """(dy비율, 눌림, 재생시간배수) 목록을 변형 파라미터로 바꾼다."""
    out = []
    for r, q, hold in keys:
        sx, sy = _squash(q)
        out.append(dict(dy=amp * r, sx=sx, sy=sy, anchor=(0.5, 1.0), hold=hold))
    return out


# ============================================================ 순환형 모션


def _sq(q: float, **kw) -> dict:
    """눌림 q 를 가로세로 배율로 바꿔 변형 파라미터 dict 로 만든다."""
    sx, sy = _squash(q)
    kw.setdefault("anchor", (0.5, 1.0))
    return dict(sx=sx, sy=sy, **kw)


def _cyc(n: int, fn) -> List[dict]:
    """
    순환형 모션 공통 틀. fn(t) 가 t=0~1 구간의 변형 파라미터를 돌려준다.
    마지막 프레임(t=1)을 '포함하지 않아' 한 바퀴가 그대로 이어진다.
    """
    return [fn(i / n) for i in range(n)]


def m_breathe(n: int = 12, pct: float = 0.045) -> List[dict]:
    """숨쉬기. 미세한 상하 신축. 정지 이모티콘에 생명감만 넣는 용도."""
    return _cyc(n, lambda t: _sq(1 + pct * math.sin(2 * math.pi * t)))


def m_bounce(n: int = 12, amp: float = 40) -> List[dict]:
    """
    통통 튀기. 가장 많이 쓰이는 모션.
    상승 38% / 하강 38% / 바닥 접촉 24% 로 나눠, 마지막 프레임이 바닥에 머물게 한다.
    (예전엔 마지막 프레임이 공중에 떠 있어서 루프 이음새가 튀었다)
    """
    def f(t):
        if t < 0.38:
            h = ease_out(t / 0.38)
        elif t < 0.76:
            h = 1 - ease_in((t - 0.38) / 0.38)
        else:
            # 바닥 접촉: 빠르게 눌렸다 천천히 복원해 루프 지점에서 중립이 되게
            p = (t - 0.76) / 0.24
            return _sq(1 - 0.13 * math.sin(math.pi * p ** 0.55), dy=0)
        return _sq(1 + 0.10 * h, dy=-amp * h)
    return _cyc(n, f)


def m_nod(n: int = 12, amp: float = 20, times: int = 2) -> List[dict]:
    """
    끄덕임. 한 바퀴에 2번 끄덕이고, 아래로 빠르게 위로 천천히 — 실제 리듬이다.
    (예전엔 sin 의 음수 구간을 0으로 잘라서 절반이 완전히 멈춰 있었다)
    """
    def f(t):
        p = (t * times) % 1.0
        d = ease_in(p / 0.38) if p < 0.38 else 1 - ease_out((p - 0.38) / 0.62)
        return _sq(1 - 0.075 * d, dy=amp * d)
    return _cyc(n, f)


def m_rock(n: int = 12, deg: float = 11) -> List[dict]:
    """갸우뚱 좌우. 발바닥을 축으로 기울이고 무게중심을 반대로 살짝 민다."""
    def f(t):
        x = math.sin(2 * math.pi * t)
        return dict(rot=deg * x, dx=-2.2 * x, anchor=(0.5, 1.0))
    return _cyc(n, f)


def m_shake(n: int = 12, amp: float = 9, cycles: int = 3) -> List[dict]:
    """부들부들 떨기. 좌우 진동 + 역위상 미세 회전으로 떨림을 강조한다."""
    def f(t):
        x = math.sin(2 * math.pi * cycles * t)
        return dict(dx=amp * x, rot=-2.5 * x, anchor=(0.5, 1.0))
    return _cyc(n, f)


def m_float(n: int = 16, amp: float = 14) -> List[dict]:
    """
    부유. 상하 1주기 + 좌우 2주기로 8자를 그린다.
    (예전엔 좌우가 반주기만 돌아서 루프 지점에서 방향이 툭 바뀌었다)
    """
    def f(t):
        a = 2 * math.pi * t
        return dict(dy=-amp * math.sin(a), dx=amp * 0.55 * math.sin(2 * a),
                    rot=3.0 * math.cos(a), anchor=(0.5, 0.5))
    return _cyc(n, f)


def m_slide(n: int = 16, dist: float = 40) -> List[dict]:
    """
    좌우 미끄러짐. 기울기를 '위치'가 아니라 '속도'(cos)에 비례시킨다.
    (예전엔 가장 멀리 간 지점에서 가장 기울어 있었다. 거긴 멈추는 곳인데.)
    """
    def f(t):
        a = 2 * math.pi * t
        return dict(dx=dist * math.sin(a), rot=-8 * math.cos(a), anchor=(0.5, 1.0))
    return _cyc(n, f)


def m_spin_wobble(n: int = 16, deg: float = 26) -> List[dict]:
    """크게 갸우뚱. 어리둥절, 멍때림. 들림(dy)을 회전과 같은 주기로 맞췄다."""
    def f(t):
        a = 2 * math.pi * t
        return dict(rot=deg * math.sin(a), dy=-7 * (1 - math.cos(a)) / 2,
                    anchor=(0.5, 1.0))
    return _cyc(n, f)


# ====================================================== 키프레임형 모션
# 동작에 뚜렷한 단계가 있는 것들. '멈춤'은 프레임 복제 대신 hold로 처리한다.


def m_jump(amp: float = 95, **_) -> List[dict]:
    """
    점프. 뜸들임 → 발사 → 체공 → 낙하 → 착지 눌림 → 탄성 복귀.
    정점(hold 2.2)과 안정(hold 1.6)을 재생시간으로 처리해 프레임을 아꼈다.
    """
    return _kf([
        ( 0.00, 1.00, 1.0),   # 중립
        ( 0.00, 0.84, 1.5),   # 웅크림 — 뜸들임
        ( 0.00, 0.93, 1.0),
        (-0.34, 1.27, 1.0),   # 발사 — 길게 늘어남
        (-0.74, 1.13, 1.0),
        (-0.95, 1.03, 1.0),
        (-1.00, 1.00, 2.2),   # 정점 — 멈칫
        (-0.93, 1.05, 1.0),
        (-0.63, 1.15, 1.0),
        (-0.22, 1.21, 1.0),   # 낙하 가속
        ( 0.00, 0.76, 1.0),   # 착지 눌림
        ( 0.00, 1.09, 1.0),   # 반동
        ( 0.00, 0.95, 1.0),
        ( 0.00, 1.00, 1.6),   # 안정
    ], amp)


def m_drop(high: float = 110, **_) -> List[dict]:
    """위에서 뚝 떨어져 튕긴 뒤, 다시 위로 사라지며 루프로 이어진다."""
    return _kf([
        (-1.00, 1.10, 1.0),
        (-0.56, 1.17, 1.0),
        (-0.13, 1.23, 1.0),
        ( 0.00, 0.73, 1.0),   # 착지 눌림
        ( 0.00, 1.13, 1.0),   # 반동
        (-0.17, 1.04, 1.0),
        ( 0.00, 0.89, 1.0),
        ( 0.00, 1.00, 2.6),   # 안정 — 멈춤
        (-0.34, 1.07, 1.0),   # 다시 떠오름
        (-0.78, 1.11, 1.0),
    ], high)


def m_pop(**_) -> List[dict]:
    """뿅 등장 → 유지 → 쏙 사라짐. 순환하면 '뿅…쏙'이 반복된다."""
    out = []
    for s, hold in [(0.16, 1.0), (0.72, 1.0), (1.14, 1.0),
                    (1.00, 3.2), (0.74, 1.0), (0.32, 1.0)]:
        out.append(dict(sx=s, sy=s, anchor=(0.5, 0.95), hold=hold))
    return out


def m_zoom_punch(peak: float = 0.18, **_) -> List[dict]:
    """강조. 확 커졌다 빠르게 복귀하고 숨 고른다. '좋아!', '최고!'."""
    out = []
    for p, hold in [(0.00, 1.0), (0.72, 1.0), (1.00, 1.2), (0.78, 1.0),
                    (0.42, 1.0), (0.16, 1.0), (0.03, 2.6)]:
        s = 1 + peak * p
        out.append(dict(sx=s, sy=s, anchor=(0.5, 0.90), hold=hold))
    return out


MOTIONS: Dict[str, Callable[..., List[dict]]] = {
    "breathe": m_breathe, "bounce": m_bounce, "nod": m_nod, "rock": m_rock,
    "shake": m_shake, "float": m_float, "slide": m_slide,
    "spin_wobble": m_spin_wobble,
    "jump": m_jump, "drop": m_drop, "pop": m_pop, "zoom_punch": m_zoom_punch,
}

# 순환형(루프 전제) / 키프레임형(단계가 있는 동작)
CYCLIC = {"breathe", "bounce", "nod", "rock", "shake", "float",
          "slide", "spin_wobble"}


def motion_extent(name: str, **kwargs) -> dict:
    """
    그 모션이 캐릭터를 '얼마나 벗어나게' 하는지 미리 계산한다.

    왜 필요한가:
      점프는 캐릭터를 위로 95px 올린다. 글자 자리를 머리 위에 비워뒀는데
      그대로 점프시키면 글자와 부딪히거나 캔버스 밖으로 잘린다.
      그리기 전에 필요한 여백을 알아야 배치를 맞출 수 있다.

    돌려주는 값 (단위: 픽셀, scale 은 배율)
      up / down / left / right : 각 방향으로 최대 얼마나 벗어나는가
      scale                    : 늘어남과 '회전'까지 포함한 최대 외곽 배율

    회전을 꼭 넣어야 한다. 한 변 S 인 그림을 th 만큼 돌리면 외곽 사각형이
    S*(|cos th| + |sin th|) 로 넓어진다. 26도면 34% 커진다.
    (이걸 빠뜨려서 rock 과 spin_wobble 이 캔버스 밖으로 잘렸다)
    """
    params = MOTIONS[name](**kwargs)
    dys = [p.get("dy", 0.0) for p in params]
    dxs = [p.get("dx", 0.0) for p in params]

    spans = []
    for p in params:
        th = math.radians(p.get("rot", 0.0))
        spread = abs(math.cos(th)) + abs(math.sin(th))
        spans.append(max(p.get("sx", 1.0), p.get("sy", 1.0)) * spread)

    return {
        "up": max(0.0, -min(dys)),
        "down": max(0.0, max(dys)),
        "left": max(0.0, -min(dxs)),
        "right": max(0.0, max(dxs)),
        "scale": max(spans),
    }


def apply_motion(base: Image.Image, name: str, base_ms: int = 80,
                 scale: float = 1.0, **kwargs) -> Tuple[List[Image.Image], List[int]]:
    """
    그림 1장 + 모션 이름 → (프레임 리스트, 프레임별 재생시간 ms)

    scale : 이동량(dx, dy)만 줄이거나 키운다. 여백이 모자랄 때 움직임을
            작게 만들어 캔버스 밖으로 안 나가게 하는 용도.
            눌림·회전은 건드리지 않아 동작의 느낌은 유지된다.
    재생시간은 카카오 규격(0.05~2.0초)으로 잘라낸다.
    """
    params = MOTIONS[name](**kwargs)
    if scale != 1.0:
        params = [dict(p, dx=p.get("dx", 0.0) * scale, dy=p.get("dy", 0.0) * scale)
                  for p in params]
    frames = [transform_frame(base, **p) for p in params]
    durations = [int(min(2000, max(50, base_ms * p.get("hold", 1.0))))
                 for p in params]
    return frames, durations

## 5. 효과 모듈 — `geo_effects.py`
**담당: 강은석** — 하트·반짝임·눈물·땀·김·음표와 한글 글자를 Pillow로 직접 그린다. AI 호출 없음.

In [ ]:
%%writefile geo_effects.py
# -*- coding: utf-8 -*-
"""
효과 합성 모듈 — 하트 / 반짝임 / 눈물 / 땀 / 분노 / 음표 / 글자

AI에게 "하트를 그려줘"라고 부탁하면 매번 다르게 나오고 비용도 든다.
단순한 도형과 글자는 Pillow로 직접 그리는 게 정확하고 공짜다.

[루프 이음새 처리]
  각 효과 입자는 자기 주기의 시작과 끝에서 알파 0(완전 투명)이 된다.
  보이지 않는 상태에서 한 바퀴가 닫히므로 반복 재생해도 이음새가 없다.

[다크모드 대응]
  카카오와 OGQ 둘 다 "어두운 배경에서도 보이도록 흰색 테두리"를 요구한다.
  add_white_outline() 이 알파를 부풀려 흰 실루엣을 뒤에 깔아준다.
"""

from __future__ import annotations

import glob
import math
import os
from typing import Callable, Dict, List, Optional, Tuple

from PIL import Image, ImageDraw, ImageFilter, ImageFont

CANVAS = 360
LINE = (74, 48, 26, 255)

# ------------------------------------------------------------- 기준점 계산
# 효과를 고정 좌표에 찍으면 캐릭터 생김새가 바뀔 때 전부 틀어진다.
# (실제로 '고마워' 글자가 얼굴 한가운데를 덮는 사고가 있었다)
# 그래서 캐릭터가 실제로 차지한 영역(bbox)에서 기준점을 매번 계산한다.

_DEFAULT_BBOX = (60, 120, 300, 340)   # 기준점을 못 받았을 때의 보수적 추정


def anchors_from_bbox(bbox, canvas=(CANVAS, CANVAS)) -> Dict[str, Tuple[int, int]]:
    """
    캐릭터 bbox -> 효과 기준점들.

    bbox 는 캔버스 위에 '실제로 배치된' 캐릭터의 영역이어야 한다.
    (emoticon_tools.fit_art_set() 이 배치 결과와 함께 돌려준다)
    """
    x0, y0, x1, y1 = bbox
    w, h = max(1, x1 - x0), max(1, y1 - y0)
    cx = (x0 + x1) // 2
    cw, ch = canvas

    def clamp(p):
        return (max(0, min(cw, int(p[0]))), max(0, min(ch, int(p[1]))))

    return {
        # 머리 위 '빈 공간의 한가운데'. 글자가 여기 중앙정렬로 놓이므로
        # 캐릭터 바로 위(y0 - 조금)로 잡으면 글자 아랫부분이 귀를 덮는다.
        "above_head": clamp((cx, max(14, y0 * 0.5))),
        "below":      clamp((cx, y1 + h * 0.10)),
        "chest":      clamp((cx, y0 + h * 0.66)),
        "left_eye":   clamp((x0 + w * 0.30, y0 + h * 0.46)),
        "right_eye":  clamp((x0 + w * 0.70, y0 + h * 0.46)),
        "head_tr":    clamp((x1 - w * 0.06, y0 + h * 0.12)),
        "head_tl":    clamp((x0 + w * 0.06, y0 + h * 0.12)),
        # 캐릭터 바깥 위쪽. 음표처럼 '허공에 떠오르는' 효과용.
        # 실루엣에 겹치면 지저분해서 bbox 모서리 바깥으로 뺀다.
        "float_tr":   clamp((min(cw - 26, x1 + w * 0.02), max(20, y0 + h * 0.04))),
        "float_tl":   clamp((max(26, x0 - w * 0.02), max(20, y0 + h * 0.04))),
        "_bbox":      (x0, y0, x1, y1),
        "_size":      (w, h),
    }


def default_anchors(canvas=(CANVAS, CANVAS)) -> Dict[str, Tuple[int, int]]:
    return anchors_from_bbox(_DEFAULT_BBOX, canvas)


def _pick(anchors, name, canvas=(CANVAS, CANVAS)):
    """앵커 dict에서 기준점을 꺼낸다. 없으면 보수적 기본값."""
    if anchors is None:
        anchors = default_anchors(canvas)
    return anchors[name]


def _scale_of(anchors, canvas=(CANVAS, CANVAS)) -> float:
    """
    효과 크기 배율.

    캐릭터 크기에 비례시키되, '캔버스에 남은 여백'도 함께 본다.
    캐릭터가 화면을 꽉 채우면 효과를 놓을 자리가 없는데, 크기만 보고 키우면
    효과가 캔버스 밖으로 잘린다. (실제로 sparkle·anger·note 가 잘렸다)
    """
    if not anchors or "_size" not in anchors:
        return 1.0
    base = anchors["_size"][1] / 220.0
    x0, y0, x1, y1 = anchors["_bbox"]
    cw, ch = canvas
    room = max(8, min(y0, x0, cw - x1, ch - y1))   # 사방 여백 중 가장 좁은 곳
    return max(0.35, min(1.8, base, room / 26.0))

# ------------------------------------------------------------------- 폰트

# 폰트를 '고정 경로 목록'으로 찾으면 환경이 조금만 달라도 못 찾는다.
# (Colab에서 apt-get 이 조용히 실패해 글자가 전부 네모로 나온 적이 있다)
# 그래서 실제로 디렉터리를 뒤지고, 한글 글리프가 '진짜 있는지'까지 확인한다.

_FONT_DIRS = [
    "/usr/share/fonts", "/usr/local/share/fonts",
    os.path.expanduser("~/.fonts"), "C:/Windows/Fonts",
]
# 앞에 있을수록 우선. 나눔고딕은 OFL 라이선스라 상업 사용이 허용된다.
_FONT_PREFER = ["nanumgothicbold", "nanumgothic", "malgunbd", "malgun",
                "notosanskr", "notosanscjk", "applesdgothic", "gulim", "batang"]

# 시스템에 아무것도 없을 때 받아오는 곳 (Google Fonts, OFL)
_FONT_URL = ("https://github.com/google/fonts/raw/main/ofl/"
             "nanumgothic/NanumGothic-Regular.ttf")
_LOCAL_FONT = "NanumGothic.ttf"

_resolved_font = None          # 한 번 찾으면 재사용


def _has_korean(path: str) -> bool:
    """
    그 폰트에 한글 글리프가 실제로 있는지 확인한다.

    Pillow 는 폰트 폴백(다른 폰트에서 빌려오기)을 하지 않는다. 그래서 한글이
    없는 폰트로 찍으면 네모(두부)가 나온다. 그런데 크기는 정상으로 보고되므로
    getbbox() 만으로는 구분할 수 없다.
    여기서는 '가' 와 '쓰이지 않는 문자'의 렌더 결과를 비교한다.
    둘이 같으면 둘 다 네모라는 뜻이다.
    """
    try:
        f = ImageFont.truetype(path, 20)
        return bytes(f.getmask("가")) != bytes(f.getmask("\ue000"))
    except Exception:
        return False


def find_korean_font() -> Optional[str]:
    """시스템에서 한글이 되는 폰트를 찾는다. 없으면 None."""
    global _resolved_font
    if _resolved_font and os.path.exists(_resolved_font):
        return _resolved_font

    if os.path.exists(_LOCAL_FONT) and _has_korean(_LOCAL_FONT):
        _resolved_font = _LOCAL_FONT
        return _resolved_font

    found = []
    for d in _FONT_DIRS:
        if os.path.isdir(d):
            for ext in ("ttf", "ttc", "otf"):
                found += glob.glob(os.path.join(d, "**", f"*.{ext}"), recursive=True)

    def rank(p):
        name = os.path.basename(p).lower().replace("-", "").replace("_", "")
        for i, key in enumerate(_FONT_PREFER):
            if key in name:
                return i
        return len(_FONT_PREFER)

    # 이름이 익숙한 것부터, 그다음 나머지를 (너무 오래 걸리지 않게) 일부만
    ordered = sorted(found, key=rank)
    for p in ordered[:80]:
        if _has_korean(p):
            _resolved_font = p
            return p
    return None


def ensure_korean_font(download: bool = True) -> Optional[str]:
    """
    한글 폰트를 확보한다. 시스템에 없으면 나눔고딕을 받아온다.
    Colab 에서 apt-get 이 실패해도 이쪽으로 복구된다.
    """
    global _resolved_font
    p = find_korean_font()
    if p or not download:
        return p
    try:
        import urllib.request
        urllib.request.urlretrieve(_FONT_URL, _LOCAL_FONT)
        if _has_korean(_LOCAL_FONT):
            _resolved_font = _LOCAL_FONT
            return _resolved_font
    except Exception:
        pass
    return None


def load_font(size: int) -> ImageFont.FreeTypeFont:
    """한글이 되는 폰트를 불러온다. 처음 쓸 때 없으면 받아온다."""
    path = ensure_korean_font()
    if path:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()


def font_available() -> bool:
    """한글 폰트를 쓸 수 있는지. 없으면 받아와서라도 확보한다."""
    return ensure_korean_font() is not None


# --------------------------------------------------- 다크모드 흰 테두리

def add_white_outline(img: Image.Image, width: int = 5) -> Image.Image:
    """
    알파를 부풀려 흰색 실루엣을 뒤에 깔아준다.
    어두운 배경(다크모드 채팅방)에서도 형체가 보이게 하는 필수 처리.
    """
    img = img.convert("RGBA")
    alpha = img.getchannel("A")
    grown = alpha
    for _ in range(max(1, width // 2)):
        grown = grown.filter(ImageFilter.MaxFilter(5))
    grown = grown.point(lambda v: 255 if v > 40 else 0)

    halo = Image.new("RGBA", img.size, (255, 255, 255, 0))
    halo.putalpha(grown)
    return Image.alpha_composite(halo, img)


# ------------------------------------------------------------- 도형 그리기

def _heart_polygon(cx: float, cy: float, r: float) -> List[Tuple[float, float]]:
    """매개변수 하트 곡선. 폴리곤 하나로 그려서 외곽선이 한 번만 나온다."""
    pts = []
    for k in range(40):
        t = 2 * math.pi * k / 40
        x = 16 * math.sin(t) ** 3
        y = 13 * math.cos(t) - 5 * math.cos(2 * t) - 2 * math.cos(3 * t) - math.cos(4 * t)
        pts.append((cx + x * r / 17, cy - y * r / 17))
    return pts


def _star_polygon(cx: float, cy: float, r: float, inner: float = 0.30,
                  points: int = 4) -> List[Tuple[float, float]]:
    """반짝임. 4갈래 별."""
    pts = []
    for k in range(points * 2):
        ang = math.pi * k / points - math.pi / 2
        rad = r if k % 2 == 0 else r * inner
        pts.append((cx + rad * math.cos(ang), cy + rad * math.sin(ang)))
    return pts


def _drop_polygon(cx: float, cy: float, r: float) -> List[Tuple[float, float]]:
    """물방울. 위는 뾰족하고 아래는 둥글다."""
    pts = [(cx, cy - r * 1.6)]
    for k in range(1, 24):
        t = math.pi * k / 23
        pts.append((cx + r * math.sin(t) * (0.25 + 0.75 * math.sin(t) ** 0.5),
                    cy + r * (1 - math.cos(t)) - r * 0.35))
    for k in range(23, 0, -1):
        t = math.pi * k / 23
        pts.append((cx - r * math.sin(t) * (0.25 + 0.75 * math.sin(t) ** 0.5),
                    cy + r * (1 - math.cos(t)) - r * 0.35))
    return pts


def _note_shape(d: ImageDraw.ImageDraw, cx: float, cy: float, r: float,
                fill, outline, w: int) -> None:
    """음표. 머리 + 기둥 + 깃발."""
    d.ellipse([cx - r * 0.62, cy - r * 0.45, cx + r * 0.32, cy + r * 0.45],
              fill=fill, outline=outline, width=w)
    d.line([(cx + r * 0.30, cy + r * 0.18), (cx + r * 0.30, cy - r * 1.45)],
           fill=outline, width=max(3, w + 1))
    d.line([(cx + r * 0.30, cy - r * 1.45), (cx + r * 0.95, cy - r * 1.05)],
           fill=outline, width=max(3, w + 1))


SAFE = 3          # 경계에 딱 붙지 않도록 두는 여유 (px)


def _rise_room(ay: float, half: float, floor: float = 0.0) -> float:
    """
    위로 떠오르는 효과(하트·음표)가 캔버스 밖으로 나가지 않을 상승 높이.

    half 는 그 효과가 중심에서 위로 얼마나 뻗는지(반지름 + 기둥 등)다.
    이걸 빼지 않으면 '중심'은 캔버스 안인데 '그림'은 잘린다.
    floor 를 크게 주면 안전치를 덮어써서 오히려 잘리므로 기본값은 0이다.
    """
    return max(floor, min(ay - half - SAFE, ay))


def _clamp_x(x: float, half: float, cw: int) -> float:
    """좌우로도 잘리지 않게 중심 x 를 캔버스 안으로 밀어 넣는다."""
    return max(half + SAFE, min(cw - half - SAFE, x))


def _fade(p: float) -> float:
    """0에서 시작해 중간에 최대, 1에서 다시 0. 루프 이음새를 없애는 핵심."""
    return math.sin(math.pi * max(0.0, min(1.0, p)))


def _a(color: Tuple[int, int, int], alpha: float) -> Tuple[int, int, int, int]:
    return (color[0], color[1], color[2], int(max(0, min(255, alpha * 255))))


# ================================================================== 효과들
# 각 효과는 프레임 수 n을 받아 n장의 투명 오버레이를 돌려준다.


def _particles(n: int, count: int, canvas, draw) -> List[Image.Image]:
    """
    효과 6종이 공유하는 뼈대. n프레임 × count개 입자를 찍어낸다.

    입자 k는 자기 위상 p = (i/n + k/count) % 1 을 따라 0 -> 1 로 진행하고,
    알파는 _fade() 로 양 끝에서 0이 된다. 보이지 않는 상태에서 한 바퀴가
    닫히므로 반복 재생해도 이음새가 생기지 않는다.

    draw(d, k, p, al) 가 입자 하나를 그린다.
    """
    out = []
    for i in range(n):
        layer = Image.new("RGBA", canvas, (0, 0, 0, 0))
        d = ImageDraw.Draw(layer)
        for k in range(count):
            p = (i / n + k / count) % 1.0
            al = _fade(p)
            if al >= 0.02:
                draw(d, k, p, al)
        out.append(layer)
    return out


def fx_heart(n: int = 12, count: int = 3, anchors=None,
             canvas=(CANVAS, CANVAS)) -> List[Image.Image]:
    """
    설렘. 하트가 캐릭터 '양옆'에서 떠올라 흔들리며 사라진다.

    가슴 높이에서 똑바로 올리면 얼굴 한가운데를 가로지른다.
    (머리만 있는 캐릭터면 가슴 좌표 자체가 얼굴 안이다)
    그래서 실루엣 바깥에서 띄운다.
    """
    PINK = (255, 96, 128)
    x0, y0, x1, y1 = (anchors or default_anchors(canvas))["_bbox"]
    sc = _scale_of(anchors, canvas)
    pad = 18 * sc
    r_max = 20 * sc
    drift_max = (10 + 6 * (count // 2)) * sc
    start_y = y0 + (y1 - y0) * 0.55
    rise = _rise_room(start_y, r_max * 1.2, floor=50.0)
    sides = [_clamp_x(x0 - pad, r_max + drift_max, canvas[0]),
             _clamp_x(x1 + pad, r_max + drift_max, canvas[0])]

    def draw(d, k, p, al):
        base_x = sides[k % 2]
        drift = (10 + 6 * (k // 2)) * sc * math.sin(p * 4 + k)
        cx = base_x + drift * (1 if k % 2 else -1)
        d.polygon(_heart_polygon(cx, start_y - rise * p, (12 + 8 * p) * sc),
                  fill=_a(PINK, al), outline=_a(LINE[:3], al),
                  width=max(2, int(3 * sc)))

    return _particles(n, count, canvas, draw)


def fx_sparkle(n: int = 12, count: int = 4, anchors=None,
               canvas=(CANVAS, CANVAS)) -> List[Image.Image]:
    """
    최고 / 뿌듯. 반짝임이 캐릭터 '바깥쪽'에서 번쩍인다.
    (얼굴 위에 찍히지 않도록 bbox 바깥에 배치한다)
    """
    GOLD = (255, 214, 64)
    x0, y0, x1, y1 = (anchors or default_anchors(canvas))["_bbox"]
    sc = _scale_of(anchors, canvas)
    pad = 14 * sc
    r_max = 25 * sc
    cw, chh = canvas
    raw = [(x0 - pad, y0 + (y1 - y0) * 0.18), (x1 + pad, y0 + (y1 - y0) * 0.10),
           (x0 + (x1 - x0) * 0.22, y0 - pad), (x0 + (x1 - x0) * 0.78, y0 - pad * 0.6)]
    spots = [(_clamp_x(sx, r_max, cw),
              max(r_max + SAFE, min(chh - r_max - SAFE, sy)))
             for sx, sy in raw]

    def draw(d, k, p, al):
        cx, cy = spots[k % len(spots)]
        d.polygon(_star_polygon(cx, cy, (9 + 16 * al) * sc),
                  fill=_a(GOLD, al), outline=_a(LINE[:3], al), width=2)

    return _particles(n, min(count, len(spots)), canvas, draw)


def fx_tear(n: int = 12, anchors=None, canvas=(CANVAS, CANVAS)) -> List[Image.Image]:
    """슬픔. 양쪽 눈에서 눈물이 흐른다."""
    BLUE = (120, 196, 255)
    eyes = (_pick(anchors, "left_eye", canvas), _pick(anchors, "right_eye", canvas))
    sc = _scale_of(anchors, canvas)

    def draw(d, k, p, al):
        ex, ey = eyes[k]
        d.polygon(_drop_polygon(ex, ey + 76 * sc * p, (9 + 3 * p) * sc),
                  fill=_a(BLUE, al ** 0.5), outline=_a(LINE[:3], al ** 0.5), width=2)

    return _particles(n, 2, canvas, draw)


def fx_sweat(n: int = 12, anchors=None, canvas=(CANVAS, CANVAS)) -> List[Image.Image]:
    """당황. 땀방울이 머리 양옆에서 튀어 날아간다."""
    BLUE = (150, 210, 255)
    srcs = ((_pick(anchors, "head_tr", canvas), 1),
            (_pick(anchors, "head_tl", canvas), -1))
    sc = _scale_of(anchors, canvas)

    cw, chh = canvas
    r_max = 12 * sc

    def draw(d, k, p, al):
        (sx, sy), sgn = srcs[k]
        cx = _clamp_x(sx + sgn * 44 * sc * p, r_max, cw)
        cy = max(r_max * 1.8, min(chh - r_max, sy - 26 * sc * p + 58 * sc * p * p))
        d.polygon(_drop_polygon(cx, cy, (10 - 2 * p) * sc),
                  fill=_a(BLUE, al), outline=_a(LINE[:3], al), width=2)

    return _particles(n, 2, canvas, draw)


def fx_anger(n: int = 12, anchors=None, canvas=(CANVAS, CANVAS)) -> List[Image.Image]:
    """화남. 김이 머리 위로 뿜어져 올라간다. (양옆 2곳 × 2덩이 = 입자 4개)"""
    GRAY = (228, 228, 236)
    srcs = (_pick(anchors, "head_tl", canvas), _pick(anchors, "head_tr", canvas))
    sc = _scale_of(anchors, canvas)

    cw, chh = canvas
    r_max = 28 * sc

    def draw(d, k, p, al):
        sx, sy = srcs[k % 2]
        r = (11 + 17 * p) * sc
        lift = min(54 * sc, _rise_room(sy, r_max, floor=10.0))
        cx = _clamp_x(sx + (8 if k % 2 else -8) * sc * p, r_max, cw)
        cy = sy - lift * p
        d.ellipse([cx - r, cy - r * 0.78, cx + r, cy + r * 0.78],
                  fill=_a(GRAY, al * 0.92), outline=_a(LINE[:3], al * 0.74), width=2)

    return _particles(n, 4, canvas, draw)


def fx_note(n: int = 12, count: int = 2, anchors=None,
            canvas=(CANVAS, CANVAS)) -> List[Image.Image]:
    """신남. 음표가 캐릭터 오른쪽 위 허공으로 떠오른다."""
    PURPLE = (160, 128, 255)
    ax, ay = _pick(anchors, "float_tr", canvas)
    sc = _scale_of(anchors, canvas) * 1.25                      # 작게 보여서 조금 키운다
    r_max = 19 * sc
    sway = 16 * sc
    # 음표는 머리 위로 기둥과 깃발이 더 올라간다 -> 반지름의 1.7배를 여유로 본다
    top_need = r_max * 1.7 + SAFE
    ay = max(top_need, ay)                       # 시작점부터 안으로
    rise = max(0.0, min(96.0 * sc, _rise_room(ay, r_max * 1.7)))
    ax = _clamp_x(ax, r_max + sway, canvas[0])

    def draw(d, k, p, al):
        _note_shape(d, ax + sway * math.sin(p * 4 + k * 2), ay - rise * p,
                    (15 + 4 * p) * sc, _a(PURPLE, al), _a(LINE[:3], al), 2)

    return _particles(n, count, canvas, draw)


def fx_text(text: str, n: int = 12, anchors=None, canvas=(CANVAS, CANVAS),
            size: int = 0, color=(255, 255, 255), bob: float = 10,
            place: str = "above") -> List[Image.Image]:
    """
    글자 효과. 'ㅋㅋㅋ', 'Zzz', '?', '!', '고마워' 등.

    AI에게 글자를 그리게 하면 획이 깨지지만, 폰트로 직접 쓰면 100% 정확하다.
    다크모드 대응 흰 테두리(stroke)도 여기서 정확히 넣는다.

    size=0 이면 '남은 여백에 들어가는 최대 크기'를 스스로 계산한다.
    place: "above"(머리 위) | "below"(발 아래)
    """
    cw, ch = canvas
    ax, ay = _pick(anchors, "above_head" if place == "above" else "below", canvas)

    # 여백에 맞는 글자 크기를 정한다.
    # 글자는 ay 를 '중심'으로 그려지므로 위아래로 절반씩 퍼진다.
    # 렌더 높이의 절반은 대략 size*0.62 (글자높이 + 외곽선).
    if size <= 0:
        room = ay if place == "above" else (ch - ay)
        size = int(max(20, min(76, (room - 8) / 0.62)))
    # 글자가 작아질 대로 작아져도 자리가 모자라면 중심을 안으로 민다.
    # 위아래로 까딱이는 폭(bob)과 10% 확대까지 더해서 잡아야 안 잘린다.
    half = size * 0.62 * 1.10 + abs(bob) + 4 + SAFE
    ay = max(half, min(ch - half, ay))

    font = load_font(size)
    stroke = max(3, size // 12)

    out = []
    for i in range(n):
        layer = Image.new("RGBA", canvas, (0, 0, 0, 0))
        d = ImageDraw.Draw(layer)
        p = i / n
        s = 1 + 0.10 * math.sin(2 * math.pi * p)
        dy = bob * math.sin(2 * math.pi * p)
        d.text((ax, ay + dy), text, font=font, anchor="mm",
               fill=color + (255,), stroke_width=stroke, stroke_fill=LINE)
        if abs(s - 1) > 1e-6:
            layer = layer.resize((int(cw * s), int(ch * s)), Image.LANCZOS)
            sheet = Image.new("RGBA", (cw, ch), (0, 0, 0, 0))
            sheet.paste(layer, ((cw - layer.size[0]) // 2, (ch - layer.size[1]) // 2))
            layer = sheet
        out.append(layer)
    return out


EFFECTS: Dict[str, Callable[..., List[Image.Image]]] = {
    "heart": fx_heart,
    "sparkle": fx_sparkle,
    "tear": fx_tear,
    "sweat": fx_sweat,
    "anger": fx_anger,
    "note": fx_note,
}


# ---------------------------------------------------------------- 합성

def composite(frames: List[Image.Image],
              overlays: List[Image.Image]) -> List[Image.Image]:
    """
    모션이 끝난 프레임 위에 효과를 얹는다.
    효과는 캐릭터와 함께 눌리거나 기울어지면 안 되므로 변형 '이후'에 합성한다.
    프레임 수가 다르면 효과 쪽을 비율로 맞춰 집는다.
    """
    out = []
    m = len(overlays)
    for i, f in enumerate(frames):
        ov = overlays[int(i * m / len(frames))]
        out.append(Image.alpha_composite(f.convert("RGBA"), ov))
    return out

## 6. 테스트용 캐릭터 — `charsample.py`
API 없이 파이프라인을 점검할 때 쓰는 합성 캐릭터.

In [ ]:
%%writefile charsample.py
# -*- coding: utf-8 -*-
"""테스트용 합성 캐릭터 (AI가 그릴 자리의 대역)와 모션 계획표."""

from PIL import Image, ImageDraw

CANVAS = 360

# 캐릭터 기준점 (효과를 어디에 붙일지)
ABOVE_HEAD = (180, 104)
LEFT_EYE = (138, 252)
RIGHT_EYE = (222, 252)
HEAD_RIGHT = (296, 150)
HEAD_LEFT = (64, 150)
BELLY = (180, 300)


def draw_character() -> Image.Image:
    """
    위쪽에 점프 여유를 두고 배치한다.
    평평한 색 + 굵은 외곽선이라 실제 이모티콘 화풍의 변형 특성을 그대로 본다.
    """
    img = Image.new("RGBA", (CANVAS, CANVAS), (0, 0, 0, 0))
    d = ImageDraw.Draw(img)

    BODY, LINE = (255, 206, 110, 255), (82, 54, 28, 255)
    DARK, BLUSH = (60, 42, 26, 255), (255, 142, 148, 255)

    for cx in (118, 242):                                   # 귀
        d.polygon([(cx - 26, 182), (cx + 26, 182), (cx, 124)],
                  fill=BODY, outline=LINE, width=7)
    d.ellipse([70, 160, 290, 334], fill=BODY, outline=LINE, width=8)   # 몸통
    for cx in (138, 222):                                   # 눈
        d.ellipse([cx - 15, 218, cx + 15, 258], fill=DARK)
        d.ellipse([cx - 4, 226, cx + 6, 238], fill=(255, 255, 255, 255))
    for cx in (104, 256):                                   # 볼
        d.ellipse([cx - 19, 266, cx + 19, 288], fill=BLUSH)
    d.arc([164, 262, 196, 292], start=0, end=180, fill=LINE, width=7)  # 입
    d.arc([276, 258, 330, 320], start=250, end=40, fill=LINE, width=9) # 꼬리
    return img


# (모션, 기준 재생시간 ms)  — 프레임 수는 각 모션이 스스로 정한다
PLAN = [
    ("breathe",     90),
    ("bounce",      70),
    ("nod",         80),
    ("rock",        90),
    ("shake",       55),
    ("float",      100),
    ("zoom_punch",  70),
    ("pop",         70),
    ("drop",        65),
    ("jump",        62),
    ("slide",       80),
    ("spin_wobble", 85),
]

## 7. Streamlit 앱 — `app.py`
**담당: 이은석** — 화면 구성, 업로드 이미지 전달, tool calling 루프 통합

In [ ]:
%%writefile app.py
# -*- coding: utf-8 -*-
"""
딸깍콘(DdalkakCon) - Streamlit 앱 (프론트엔드 + 코드 통합)
담당: 프론트엔드 및 코드 통합

핵심 함수는 emoticon_tools.py 에 있고, 이 파일은
  - 화면 구성(사이드바 / 채팅 / CSS)
  - 업로드 이미지를 tool 쪽으로 전달
  - LangChain tool calling 루프 통합
을 담당한다.
"""

import base64
import os

import streamlit as st
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI

import emoticon_tools as et
from emoticon_tools import TOOLS, TOOL_DICT

# gpt-4o-mini 는 한국어가 자주 번역투로 무너졌다
# ("맛있고 매력적인 캐릭터네요", "방해만 좀 주세요!" 같은 문장이 나왔다).
# 같은 요청으로 비교해보니 gpt-5.4-mini 가 확실히 자연스럽고,
# 스트리밍·도구 호출 속도는 거의 같았다 (1.5초).
llm = ChatOpenAI(model="gpt-5.4-mini")
llm_with_tools = llm.bind_tools(TOOLS)


# ----------------------------------------------------- AI 응답 처리

def get_ai_response(messages):
    """LLM 응답을 스트리밍하고, tool 호출 요청이 있으면 실행한 뒤 다시 호출한다."""
    response = llm_with_tools.stream(messages)

    # 스트리밍된 AIMessageChunk들을 하나의 완성된 메시지로 합치기 위한 변수
    gathered = None

    for chunk in response:
        # 화면에는 텍스트만 흘려보낸다 (chunk 객체를 그대로 주면 repr이 찍힌다)
        if chunk.content:
            yield chunk.content if isinstance(chunk.content, str) else str(chunk.content)

        gathered = chunk if gathered is None else gathered + chunk

    if gathered is None:
        return

    # 완성된 메시지에 Tool Call이 있다면 실제 Tool을 실행
    if gathered.tool_calls:
        st.session_state.messages.append(gathered)

        for tool_call in gathered.tool_calls:
            selected_tool = TOOL_DICT[tool_call["name"]]

            with st.spinner("🎨 이모티콘 프레임을 생성하고 있어요... (30~60초)"):
                tool_msg = selected_tool.invoke(tool_call)

            st.session_state.messages.append(tool_msg)

        # ToolMessage까지 추가된 새 대화 기록으로 모델을 다시 호출해 최종 답변 생성
        for piece in get_ai_response(st.session_state.messages):
            yield piece


# ----------------------------------------------------- 화면 기본 설정

st.set_page_config(
    page_title="💬 딸깍콘",
    page_icon="💬",   # 로컬 파일 경로를 쓰면 세션이 바뀔 때 깨지므로 이모지로 둔다
    layout="wide",
)

st.markdown(
    """
<style>
/* 전체 앱의 배경색을 살짝 밝은 회색/아이보리 톤으로 */
.stApp { background-color: #F8F9FA; }

/* 버튼 모양과 색상 */
div.stButton > button, div.stDownloadButton > button {
    background-color: #4F46E5 !important;
    color: white !important;
    border-radius: 12px !important;
    border: none !important;
    padding: 8px 16px !important;
    font-weight: bold !important;
    transition: all 0.3s ease !important;
}
div.stButton > button:hover, div.stDownloadButton > button:hover {
    background-color: #3730A3 !important;
    transform: translateY(-2px);
    box-shadow: 0px 4px 12px rgba(79, 70, 229, 0.3) !important;
}

/* 카드 박스(st.metric) */
div[data-testid="stMetric"] {
    background-color: white !important;
    padding: 16px !important;
    border-radius: 16px !important;
    box-shadow: 0px 4px 15px rgba(0, 0, 0, 0.05) !important;
    border: 1px solid #E5E7EB !important;
}

/* 커스텀 태그 */
.highlight-title { color: #1E293B; font-size: 2.2rem; font-weight: 800; margin-bottom: 0px; }
.custom-badge {
    background-color: #E0E7FF; color: #3730A3;
    padding: 4px 12px; border-radius: 20px;
    font-size: 0.85rem; font-weight: 600;
    display: inline-block; margin-bottom: 15px;
}
</style>
""",
    unsafe_allow_html=True,
)


# ----------------------------------------------------- 사이드바

with st.sidebar:
    st.header("🤔 이모티콘으로 만들 이미지")
    st.divider()

    uploaded_file = st.file_uploader(
        "캐릭터 이미지 업로드 (선택)", type=["png", "jpg", "jpeg", "webp"]
    )

    if uploaded_file is not None:
        st.image(uploaded_file, caption="업로드된 이미지", use_container_width=True)
        # 업로드 이미지를 tool이 꺼내 쓸 수 있도록 등록한다.
        # Streamlit은 매 상호작용마다 스크립트를 위에서부터 다시 실행하므로
        # 여기서 등록해두면 아래 채팅 처리 시점에 항상 최신 이미지가 들어가 있다.
        et.set_reference_image(uploaded_file.getvalue(), uploaded_file.type)
    else:
        et.set_reference_image(None)
        st.caption("이미지를 안 올려도 설명만으로 캐릭터를 새로 그려줘요!")

    st.divider()
    st.header("🎁 완성된 이모티콘")

    if os.path.exists(et.GIF_PATH):
        st.image(et.GIF_PATH, caption=f"{et.CANVAS}x{et.CANVAS} 무한루프 이모티콘",
                 use_container_width=True)

        info = et.last_result
        if info:
            c1, c2 = st.columns(2)
            c1.metric("총 프레임", f"{info.get('total_frames', '-')}장")
            c2.metric("재생 간격", f"{info.get('duration_ms', '-')}ms")
            st.caption(
                f"원본 {info.get('unique_frames', '-')}장 → "
                f"{info.get('loop_mode', '-')} 루프 / "
                f"GIF {info.get('gif_size', 0) / 1024:.0f}KB"
            )

        with open(et.GIF_PATH, "rb") as file:
            st.download_button(
                label="📥 제안용 다운로드 (.gif)",
                data=file.read(),
                file_name="ddalkakcon_emoticon.gif",
                mime="image/gif",
                use_container_width=True,
            )

        if os.path.exists(et.WEBP_PATH):
            with open(et.WEBP_PATH, "rb") as file:
                st.download_button(
                    label="📥 승인 후 제출용 (.webp)",
                    data=file.read(),
                    file_name="ddalkakcon_emoticon.webp",
                    mime="image/webp",
                    use_container_width=True,
                )

        frame_paths = [p for p in et.last_result.get("frame_paths", []) if os.path.exists(p)]
        if frame_paths:
            with st.expander(f"🖼️ 생성된 원본 프레임 {len(frame_paths)}장 보기"):
                st.image(frame_paths, width=110)
    else:
        st.info("💡 채팅창에서 이모티콘 생성을 요청하면 이곳에 미리보기와 다운로드 버튼이 나타납니다!")


# ----------------------------------------------------- 본문 헤더

st.markdown('<div class="custom-badge">🤣 딸깍!으로 만드는 이모티콘</div>',
            unsafe_allow_html=True)
st.markdown('<h1 class="highlight-title">💬 딸깍콘</h1>', unsafe_allow_html=True)
st.caption(
    "캐릭터 이미지와 설명을 주면 → 기획 → 프레임 생성 → 투명배경 정리 → "
    f"{et.CANVAS}×{et.CANVAS} 무한루프 이모티콘 파일까지 한 번에 만들어 드려요."
)


# ----------------------------------------------------- 시스템 프롬프트

SYSTEM_PROMPT = """[Role & Goal]
당신은 사용자의 요청을 바탕으로 세심하고 매력적인 '움직이는 이모티콘(Animated Emoticon)'을 기획하고, create_animated_emoticon 도구로 실제 파일까지 만들어 주는 친절한 AI 전문 디자이너입니다.

[Language]
- 반드시 자연스러운 한국어로 답합니다.
- 영어를 직역한 어색한 문장을 쓰지 않습니다.
  (나쁜 예: "팔이 땅에 닿는 입장을 가장 선호하는 것입니다", "완전한 토지를 소유",
   "방해만 좀 주세요", "맛있고 매력적인 캐릭터")
- 문장이 어색하면 다시 쓰고, 뜻이 통하는지 확인한 뒤 답합니다.

[Persona & Tone]
- 항상 밝고 다정하며 친근한 어조로 대화합니다. (예: "~해요!", "~해드릴게요 😊")
- 사용자의 아이디어를 적극적으로 칭찬하고 긍정적인 반응을 보입니다.
- 전문 용어보다는 누구나 이해하기 쉽고 따뜻한 표현을 씁니다.

[Core Instructions]
1. 사용자 프롬프트 최우선 반영
   - 캐릭터 외형, 동작, 감정, 색상, 분위기, 텍스트 등 사용자가 준 조건을 하나도 놓치지 않습니다.
   - 요청이 구체적이지 않으면 의도를 해치지 않는 선에서 매력적인 연출을 더해줍니다.

2. 먼저 기획안을 제시합니다
   - 캐릭터 디자인(외형/색상/분위기), 주요 감정과 표정, 프레임별 자세, 시각 효과를 설명합니다.
   - 프레임은 '시작 자세 → 핵심 동작 → 돌아오기' 흐름으로 6~8단계를 묘사합니다.

3. 도구 호출 규칙 (중요)
   - 사용자가 "만들어줘", "생성해줘", "좋아 이걸로 해줘" 등 제작을 요청하면
     반드시 create_animated_emoticon 도구를 호출합니다.
   - 사용자가 기획안만 보고 싶어하면 도구를 호출하지 말고 대화로만 답합니다.
   - 도구에 넘기는 character_description 과 frame_steps 는 반드시 '영문'으로 작성합니다.
     이미지 생성 AI가 영문 프롬프트에서 품질이 더 좋기 때문입니다.
   - frame_steps 의 각 문장은 '한 장의 정지 그림'을 묘사합니다. 움직임을 설명하지 않고
     그 순간의 자세와 표정만 적습니다. 배경이나 화면 구도는 언급하지 않습니다
     (도구가 투명배경과 규격을 알아서 처리합니다).
   - 반복 재생되므로 첫 번째와 마지막 프레임은 같은 기본 자세로 둡니다.
     마지막 자세가 썸네일이 되니 캐릭터를 가장 잘 보여주는 자세로 합니다.
   - 효과(하트·반짝임·눈물·땀·김·음표)와 한글 글자는 frame_steps 에 쓰지 말고
     effect 와 overlay_text 인자로 넘깁니다. 그래야 정확한 위치에 또렷하게 들어갑니다.

4. 도구 실행 후
   - 결과 메시지를 바탕으로 무엇이 만들어졌는지 친절하게 요약합니다.
   - 왼쪽 사이드바에서 미리보기와 다운로드가 가능하다고 안내합니다.
   - 수정하고 싶은 점이 있으면 말해달라고 덧붙입니다.
"""

if "messages" not in st.session_state:
    st.session_state["messages"] = [SystemMessage(content=SYSTEM_PROMPT)]


# ----------------------------------------------------- 대화 기록 출력

def display_user_content(content):
    """HumanMessage는 텍스트만, 또는 텍스트+이미지 리스트일 수 있다."""
    if isinstance(content, str):
        st.write(content)
        return
    if isinstance(content, list):
        for item in content:
            if not isinstance(item, dict):
                continue
            if item.get("type") == "text":
                st.write(item.get("text", ""))
            elif item.get("type") == "image_url":
                img_url = item.get("image_url", {}).get("url", "")
                if img_url:
                    st.image(img_url, caption="첨부한 이미지", width=200)


for msg in st.session_state.messages:
    if isinstance(msg, SystemMessage):
        continue

    elif isinstance(msg, AIMessage):
        if msg.content:
            st.chat_message("assistant").write(msg.content)

    elif isinstance(msg, HumanMessage):
        with st.chat_message("user"):
            display_user_content(msg.content)

    elif isinstance(msg, ToolMessage):
        with st.chat_message("assistant", avatar="🛠️"):
            with st.expander("이모티콘 생성 도구 실행 결과"):
                st.code(msg.content, language=None)


# ----------------------------------------------------- 사용자 입력 처리

if prompt := st.chat_input("예: 이 캐릭터가 신나서 점프하는 이모티콘 만들어줘!"):
    if uploaded_file is not None:
        base64_image = base64.b64encode(uploaded_file.getvalue()).decode("utf-8")
        user_content = [
            {"type": "text", "text": prompt},
            {
                "type": "image_url",
                "image_url": {"url": f"data:{uploaded_file.type};base64,{base64_image}"},
            },
        ]
        user_message = HumanMessage(content=user_content)
    else:
        user_message = HumanMessage(content=prompt)

    with st.chat_message("user"):
        st.write(prompt)

    st.session_state.messages.append(user_message)

    with st.chat_message("assistant"):
        result = st.write_stream(get_ai_response(st.session_state["messages"]))

    if result:
        st.session_state["messages"].append(AIMessage(content=result))

    # 사이드바가 새로 만들어진 이모티콘을 바로 보여줄 수 있도록 다시 그린다
    st.rerun()

## 8. 자체 점검 (API 비용 0원)

이미지 생성 없이 모션·효과·규격이 제대로 도는지 먼저 확인한다.
**여기서 통과하면, 문제가 생겨도 원인이 '이미지 생성' 쪽으로 좁혀진다.**

In [ ]:
# ============================================================
#  파이프라인 자체 점검 (API 호출 0회 / 비용 0원)
#  이미지 처리와 모션이 제대로 도는지 먼저 확인한다.
# ============================================================
import importlib
import emoticon_tools as et, geo_motion as gm, geo_effects as fx, charsample
for m in (et, gm, fx, charsample):
    importlib.reload(m)
from PIL import Image

base = charsample.draw_character()

# (1) 모션 12종 + 효과: 프레임 수와 용량이 규격 안에 드는지
print(f"{'모션':<13}{'프레임':>6}{'길이':>8}{'용량':>10}")
print("-" * 40)
for name, ms in charsample.PLAN:
    frames, durs = gm.apply_motion(base, name, base_ms=ms)
    size = et.save_gif(frames, path=f"/content/_chk_{name}.gif", duration_ms=durs)
    flag = "" if (len(frames) <= 24 and size <= et.SAFE_MAX_BYTES) else "  <-- 초과"
    print(f"{name:<13}{len(frames):>6}{sum(durs)/1000:>7.2f}s{size/1024:>9.1f}KB{flag}")

# (2) 한글 폰트
print()
print("한글 폰트:", "사용 가능" if fx.font_available() else "없음 (1번 셀 재실행)")

# (3) 눈으로 확인
frames, durs = gm.apply_motion(base, "jump", base_ms=62)
ov = fx.fx_text(n=len(frames), text="고마워", size=52)
merged = fx.composite(frames, ov)
et.save_gif(merged, path="/content/_chk_demo.gif", duration_ms=durs)
from IPython.display import Image as IPImage, display
display(IPImage(filename="/content/_chk_demo.gif"))

## 9. 통합 테스트 — 실제 이미지 생성까지

**확인하려는 것**
1. 이 API 키로 어떤 이미지 모델이 열려 있는가
2. 업로드 이미지에서 캐릭터가 유지되는가
3. 3종이 **같은 캐릭터**로 나오는가 (일관성)
4. 투명 배경이 깨끗한가 (구멍/노이즈 통계로 측정)
5. **Pillow로 그린 효과가 AI 화풍과 어울리는가** ← 눈으로 판정
6. 규격을 만족하는가

In [ ]:
%%writefile test_integration.py
# -*- coding: utf-8 -*-
"""
딸깍콘 (다) 통합 테스트 — 실제 OpenAI 이미지 생성 + 기하 변형 + 효과 합성

이 테스트로 확인하는 것
  1. 이 API 키로 어떤 이미지 모델이 열려 있는가        -> 모델 점검 (과금 없음)
  2. 업로드 이미지에서 캐릭터가 유지되는가              -> 기준 원화
  3. 종별 포즈가 '같은 캐릭터'로 나오는가               -> 일관성
  4. 투명 배경이 깨끗한가                              -> 투명%, 메운 구멍
  5. Pillow 효과가 AI 화풍과 어울리는가                -> 효과 침범률 + 눈으로
  6. 플랫폼 규격을 만족하는가                          -> 프레임 수, 용량

API 호출: 기준 원화 1회 + 종 수 (기본 3종 = 4회, 약 $0.04)

사용법
  python test_integration.py                 # 합성 캐릭터 레퍼런스 (준비물 없음)
  python test_integration.py 내캐릭터.png      # 내 이미지 레퍼런스
  python test_integration.py --no-ref         # 설명만으로 생성
"""

import os
import sys
import time
from concurrent.futures import ThreadPoolExecutor

HERE = os.path.dirname(os.path.abspath(__file__))
OUT = os.path.join("/content" if os.path.isdir("/content") else HERE, "integration_out")
os.makedirs(OUT, exist_ok=True)
os.environ["DDALKAK_OUT_DIR"] = OUT      # emoticon_tools 가 import 때 읽는다
sys.path.insert(0, HERE)

import numpy as np
from PIL import Image

import emoticon_tools as et
import geo_effects as fx
import geo_motion as gm

# 다른 셀이 먼저 import 했을 수도 있으니 경로를 명시적으로 덮어쓴다
et.OUT_DIR = OUT
for _n, _f in [("BASE_PATH", "generated_base.png"), ("GIF_PATH", "generated_emoticon.gif"),
               ("WEBP_PATH", "generated_emoticon.webp")]:
    setattr(et, _n, os.path.join(OUT, _f))
et.FRAME_PATH_FMT = os.path.join(OUT, "generated_frame_{:02d}.png")

CANVAS = (et.CANVAS, et.CANVAS)
RESERVE_TOP, MARGIN = 0.18, 0.06     # 효과·글자를 올릴 자리 (안 비우면 얼굴을 덮는다)
MODELS = ["gpt-image-1.5", "gpt-image-2", "gpt-image-1-mini", "gpt-image-1"]

CHARACTER = ("a round chubby cream-yellow cat character with small triangular ears, "
             "big dark round eyes, pink round cheeks, a tiny mouth and a thin curled tail")

# (종, 대표 포즈 영문 묘사, 모션, 효과, 효과인자, 기준 ms)
SPECIES = [
    ("좋아", "giving a confident thumbs up with one paw raised, bright proud smile",
     "zoom_punch", "sparkle", {}, 75),
    ("고마워", "bowing head slightly with both paws together in front of chest, "
     "warm grateful smile, eyes closed softly", "bounce", "text", {"text": "고마워"}, 75),
    ("신남", "cheering with both arms raised high, wide open happy mouth, excited",
     "jump", "note", {}, 62),
]


def step(title):
    print(f"\n{'=' * 66}\n  {title}\n{'=' * 66}")


def die(msg):
    print(f"\n{msg}")
    sys.exit(1)


# ------------------------------------------------------------- 1. 환경 점검
step("1. 환경 점검")
try:
    import openai
except ImportError:
    die("openai 패키지가 없습니다 -> pip install openai")

if not os.environ.get("OPENAI_API_KEY"):
    die("OPENAI_API_KEY 가 없습니다 -> 2번 셀을 실행하세요")

print(f"  openai {openai.__version__} / 한글 폰트 "
      f"{'OK' if fx.font_available() else '없음 (apt-get install -y fonts-nanum)'}"
      f"\n  출력 폴더: {OUT}")

# ------------------------------------------------------------- 2. 모델 점검
step("2. 모델 점검 (과금 없음)")
client = openai.OpenAI()
chosen = None
for name in MODELS:
    try:
        client.models.retrieve(name)
        print(f"  {name:<20} 사용 가능" + ("   <-- 선택" if chosen is None else ""))
        chosen = chosen or name
    except Exception as exc:
        print(f"  {name:<20} 불가 ({type(exc).__name__})")
if chosen is None:
    die("쓸 수 있는 이미지 모델이 없습니다. 기관에 열려 있는 모델명을 확인하세요.")
et.IMAGE_MODEL = chosen

# --------------------------------------------------------- 3. 레퍼런스 선택
step("3. 레퍼런스 이미지")
args = [a for a in sys.argv[1:] if not a.startswith("--")]
if "--no-ref" in sys.argv:
    et.set_reference_image(None)
    print("  레퍼런스 없음 — 설명만으로 캐릭터를 새로 생성합니다")
elif args:
    if not os.path.exists(args[0]):
        die(f"파일을 찾을 수 없습니다: {args[0]}")
    data = open(args[0], "rb").read()
    et.set_reference_image(data, "image/png" if args[0].lower().endswith(".png")
                           else "image/jpeg")
    print(f"  업로드 이미지: {args[0]} ({len(data)/1024:.0f}KB)")
else:
    from charsample import draw_character
    p = os.path.join(OUT, "00_reference.png")
    draw_character().save(p)
    et.set_reference_image(open(p, "rb").read(), "image/png")
    print(f"  합성 캐릭터를 레퍼런스로 사용 ({p})")

# ------------------------------------------------------- 4. 기준 원화 생성
step(f"4. 기준 원화 생성 ({chosen}, API 1회)")
t0 = time.time()
try:
    base = et.build_base_sheet(CHARACTER, quality="low")
except Exception as exc:
    die(f"실패: {type(exc).__name__}: {exc}")
print(f"  완료 {time.time()-t0:.1f}초, {base.size}, 저장 {et.BASE_PATH}")

# -------------------------------------------------- 5. 종별 대표 포즈 생성
step(f"5. 종별 대표 포즈 생성 (API {len(SPECIES)}회, 병렬)")
base_bytes = et._to_png_bytes(base)
t0 = time.time()


def make_pose(item):
    label, pose = item[0], item[1]
    try:
        return label, et.generate_pose(base_bytes, pose, name=f"base_{label}"), None
    except Exception as exc:
        return label, None, f"{type(exc).__name__}: {exc}"


poses = {}
with ThreadPoolExecutor(max_workers=len(SPECIES)) as pool:
    for label, img, err in pool.map(make_pose, SPECIES):
        print(f"  {label:<8} " + ("생성 완료" if img is not None else f"실패 — {err}"))
        if img is not None:
            poses[label] = img
print(f"  {len(poses)}/{len(SPECIES)}종, {time.time()-t0:.1f}초 (병렬)")
if not poses:
    die("모든 포즈 생성에 실패했습니다.")

# ------------------------------------------- 6. 투명 배경 품질 + 배치
step("6. 투명 배경 품질 + 배치")
cleaned = {}
print(f"  {'종':<8}{'생성직후 투명%':>14}{'외곽 반투명':>12}")
for label, img in poses.items():
    a = np.asarray(img.getchannel("A"))
    cleaned[label] = et.stabilize_alpha(img)
    print(f"  {label:<8}{(a < 128).mean()*100:>13.1f}%"
          f"{int(((a > 20) & (a < 235)).sum()):>12}")
print("  * 투명% 가 0에 가까우면 배경이 안 날아간 것 (프롬프트 재조정 필요)")

placed = et.fit_art_set(cleaned, canvas=CANVAS, margin=MARGIN, reserve_top=RESERVE_TOP)
print(f"\n  위쪽 {RESERVE_TOP:.0%} 를 효과·글자용으로 비움")
print(f"  {'종':<8}{'배치된 bbox':>24}{'위 여백':>8}{'메운 구멍':>10}")

art, anchors = {}, {}
for label, (img, box) in placed.items():
    before = int((np.asarray(img.getchannel("A")) < 128).sum())
    filled = et.fill_interior_holes(img)
    after = int((np.asarray(filled.getchannel("A")) < 128).sum())
    art[label] = filled
    anchors[label] = fx.anchors_from_bbox(box, CANVAS)
    filled.save(os.path.join(OUT, f"pose_{label}.png"))
    print(f"  {label:<8}{str(box):>24}{box[1]:>7}px{before-after:>10}")
print("  * 위 여백이 충분해야 글자가 얼굴을 안 덮는다")

# ------------------------------------------- 7. 모션 + 효과 + 저장
step("7. 모션 + 효과 적용 (API 호출 없음)")
print(f"  {'종':<8}{'모션':<12}{'효과':<9}{'프레임':>6}{'효과침범':>9}{'용량':>10}  판정")

for label, _pose, motion, effect, kw, ms in SPECIES:
    if label not in art:
        continue
    frames, durs = gm.apply_motion(art[label], motion, base_ms=ms)
    n = len(frames)

    # 효과 기준점은 '실제 배치된 캐릭터'에서 계산한다 (고정 좌표 금지)
    maker = fx.fx_text if effect == "text" else fx.EFFECTS[effect]
    ov = maker(n=n, anchors=anchors[label], canvas=CANVAS, **kw)

    # 효과가 캐릭터 실루엣을 덮은 비율 — 낮을수록 깔끔하다
    ch = np.asarray(art[label].getchannel("A")) > 128
    om = np.asarray(ov[n // 2].getchannel("A")) > 128
    intrude = (ch & om).sum() / max(1, om.sum()) * 100

    # 변환은 노이즈를 만들지 않으므로 프레임마다 stabilize_alpha 를 돌리지 않는다
    # (측정: 14프레임 191ms 낭비 + 반짝임 끝과 글자 획이 깎이는 부작용)
    merged = fx.composite(frames, ov)

    size, note = et.shrink_until_under_limit(
        merged, os.path.join(OUT, f"{label}.gif"), durs, limit=et.SAFE_MAX_BYTES)

    # 카카오 제출용: 번호 붙인 PNG 프레임 (WebP Animator 에 넣을 것)
    fdir = os.path.join(OUT, f"frames_{label}")
    os.makedirs(fdir, exist_ok=True)
    for i, f in enumerate(merged, 1):
        f.save(os.path.join(fdir, f"{i:02d}.png"))

    bad = [x for x in (f"프레임{n}" if n > 24 else "",
                       f"용량{size//1024}KB" if size > et.SAFE_MAX_BYTES else "",
                       note) if x]
    print(f"  {label:<8}{motion:<12}{effect:<9}{n:>6}{intrude:>8.1f}%"
          f"{size/1024:>9.1f}KB  {', '.join(bad) or 'OK'}")

# ------------------------------------------------------------- 8. 결과
step("8. 결과")
print(f"""  생성 폴더: {OUT}

  확인해 주세요
   (1) 00_reference.png / generated_base.png  캐릭터가 유지됐는지
   (2) pose_*.png                             3종이 '같은 캐릭터'인지
   (3) *.gif                                  효과가 AI 화풍과 어울리는지
   (4) frames_*/01.png~                       WebP Animator 에 넣을 프레임

  총 API 호출 {1 + len(poses)}회 (약 ${(1 + len(poses)) * 0.009:.3f})""")

In [ ]:
# ============================================================
#  (다) 통합 테스트 — 실제 이미지 생성까지 연결
#  API 호출 4회 (약 $0.04). 아래 중 하나를 골라 실행한다.
# ============================================================

# A) 합성 캐릭터를 레퍼런스로 — 준비물 없이 바로 실행
!python test_integration.py

# B) 내 캐릭터 이미지를 쓰려면 (왼쪽 파일창에 올린 뒤)
# !python test_integration.py /content/내캐릭터.png

# C) 레퍼런스 없이 설명만으로 생성
# !python test_integration.py --no-ref

In [ ]:
# 결과 확인
import glob, os
from IPython.display import Image as IPImage, display

OUT = "/content/integration_out"

print("=== 기준 원화 / 종별 대표 포즈 ===")
for p in ["00_reference.png", "generated_base.png"] + \
         [os.path.basename(x) for x in sorted(glob.glob(f"{OUT}/pose_*.png"))]:
    f = os.path.join(OUT, p)
    if os.path.exists(f):
        print(p)
        display(IPImage(filename=f, width=200))

print("=== 최종 이모티콘 ===")
for f in sorted(glob.glob(f"{OUT}/*.gif")):
    print(os.path.basename(f), f"{os.path.getsize(f)/1024:.0f}KB")
    display(IPImage(filename=f))

## 10. 서버 실행

`--theme.base light` 를 꼭 붙인다. CSS가 배경만 밝게 고정해놔서,
브라우저가 다크모드면 글자가 흰색으로 그려져 **흰 배경에 흰 글자**가 된다.
(AI 답변이 안 보이는 증상)

In [ ]:
!streamlit run app.py --server.port 8501 --theme.base light &>/content/logs.txt &

## 11. 외부 접속 터널

`https://....trycloudflare.com` 주소가 나오면 그 주소로 접속한다.
**이 셀은 실행 중 상태로 둬야 한다.** 멈추면 터널이 닫힌다.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [ ]:
!./cloudflared tunnel --url http://localhost:8501

## 12. 문제가 생길 때

```python
!tail -40 /content/logs.txt      # 서버 로그
!pkill -f streamlit              # 서버 재시작
```

| 증상 | 조치 |
|---|---|
| `SecretNotFoundError` | 2번 셀의 **노트북 액세스 토글** 확인 |
| 런타임이 끊겼다 | **1번 셀부터** 다시 실행 (설치·환경변수가 날아간다) |
| 글자가 네모(□)로 나온다 | 1번 셀의 `fonts-nanum` 설치 확인 |
| 모델을 못 찾는다 | 9번 셀의 **모델 점검** 출력을 확인. 열려 있는 모델명을 `emoticon_tools.py` 의 `IMAGE_MODEL` 에 넣는다 |
| 투명% 가 0에 가깝다 | 배경이 안 날아간 것. `STYLE_SUFFIX` 프롬프트 조정 필요 |
| 3종이 다른 캐릭터로 나온다 | `quality="medium"` 으로 올려 재시도 |
| 터널 주소가 안 열린다 | 10번 실행 후 10초 기다렸다 11번 실행 |